# HEO Na 층상 양극재 MLIP 스크리닝 v4 — dV 분해 + Na 4-point x\* (RunPod 멀티 GPU)

v3 파이프라인(엔진 v1, 물리 v2, 채점 2026-09-04 v2 = tier 0 억제 + tier 1 rank-sum(x_cross↑, Ehull_eff↑))에
**CHANGE_SPEC_v4**의 두 가지만 얹었다. 채점·관문·불변량 1–7의 정의는 바이트 단위로 v3와 같다.

| 변경 | 내용 | 산출 |
|---|---|---|
| **A** | 부피 변화를 격자 행렬로 정확 분해: 면내 dA, 기울기 면역 층간 dh_perp, von Mises 모양 변형, 층간 간격 | `dV_pct, dA_pct, dh_perp_pct, vm_strain, d_inter_pris, dV_legacy` (보고 열) |
| **B** | Na 조성 2점 → 4점 (x = 1, 22/27, 18/27, 14/27) + 전이점 x\* 구간화, 구간 전압 V_seg, x-방향 형성에너지 Ef | `dE_x081/x067, verdict_x*, x_star_*, V_seg_*, Ef_*, convexity_flag` (보고 열) |

> **스펙과 다른 사실 하나.** 스펙 A-1이 말한 "a²c 로그미분 근사"는 이 코드에 없었다. 체크포인트의 `volume`은 처음부터
> ASE `get_volume()` = |det L| 정확값이다. 따라서 `dV_legacy`(= v3 `dV_nat`)와 `dV_pct`는 정의상 같고, A의 실질은
> **분해 열**이다 — 셀11결과.md §7-5의 "등방 dV가 c축 점프를 가린다"를 정확히 푼다. 격자 행렬은 v3 체크포인트에 없어
> v3 구조는 `structures/relaxed/*.cif`에서 읽고(회전만 잃음 → det·|a×b|·tr E·von Mises 전부 회전 불변), v4 계산분부터는
> 체크포인트에 `cell` 열로 기록한다.

**실행 구조 (v3 결과 재사용)**: `HEO_WORKDIR=/workspace/heo_v4`(신규) + `HEO_BASE_WORKDIR=/workspace/heo_v2`(v3, **읽기 전용**).
워커가 base의 체크포인트·SQS·CIF를 먼저 찾으므로 앵커·캘리브레이션·PASS 1·hull·PASS 2는 전부 캐시 히트, **PASS 3(x081/x067)만 새로 돈다**.
base 디렉터리에는 어떤 단계도 쓰지 않는다 (불변량 10 + `ckpt_files()` 순서로 base 값 우선).

**Run-All이 끊기던 원인과 수정 (2026-09-10 검토)**
1. 팟 노트북에서 **셀 7b 배리어 코드 셀이 사라져** 있었다 → Run All 시 PASS 1 런처 직후 셀 8이 실행되어 커버리지 98% assert로 중단. 복원.
2. 옛 첫 셀의 `HEO_TERMINATE_MODE="stop"` 강제 → 60분 유휴에 워치독이 팟을 내렸다(셀11결과 §3-3). `none` 유지 + `HEO_IDLE_MIN` 기본 180.
3. 워커가 비정상 종료(Warp OOM 등)하면 배리어 assert로 노트북이 멈췄다 → 배리어가 **1회 자동 재기동**(체크포인트 재개) 후 재대기.
4. `pip install` 무핀 → import 선시도, 설치된 버전을 `results/pkg_versions.json`·audit에 기록.
5. 브라우저 Run All은 연결이 끊기면 출력을 잃는다(계산은 계속) → 마지막 셀의 `nohup jupyter nbconvert --execute` 경로 권장.

```
1  CONFIG (+base)  1b 워치독  2 INSTALL  3 워커 확인  4 조성 열거
5  ★ 앵커 게이트   5c ★ 캘리브레이션   5d ★ 앵커 x4 파일럿 (신규)
6  PASS 1 런처 → 7 모니터 → 7b 배리어(복원)     8 hull(LP)
9  PASS 2 런처 → 10 모니터 → 10b 배리어
10c PASS 3 타깃+런처 (신규, HEO_X4_SCOPE) → 10d 모니터 → 10e 배리어
11 집계 · 판정 · 채점(v3 그대로) · v4 열 · 불변량 1,2,4,7,9 · audit
12 top-N 내보내기   12b (선택) base 대조   12c report_v4.md (신규)   13 아카이브
```

## 셀 1 — 설정

In [1]:
import os
os.environ["HEO_TERMINATE_MODE"] = "stop"     # stop | delete | none -- cell 13 stops the pod when done; watchdog after 180 min idle
# NOTE (CLAUDE.md section 4): the MP key should come from the shell, not be written into the notebook.
# Left as the user had it; delete the line and `export MP_API_KEY=...` in the shell before sharing this file.
os.environ["MP_API_KEY"] = "0md9oV9ieH21W4cmejXD6YQZHz6rRv7F"
MP_API_KEY = os.environ["MP_API_KEY"]

In [2]:
# ============ CELL 1: CONFIG ============
# Single source of truth is heo_worker.py; this cell only sets the environment it reads.
import os, json, subprocess, sys, time, glob
from pathlib import Path
import pandas as pd

os.environ.setdefault("HEO_WORKDIR",      "/workspace/heo_v4")   # v4: a NEW workdir (Network Volume)
os.environ.setdefault("HEO_BASE_WORKDIR", "/workspace/heo_v2")   # the v3 run, mounted READ-ONLY (checkpoints, SQS, CIFs)
os.environ.setdefault("HEO_BASE_RESULTS", "/workspace/heo_v2/results/results_v3.csv")  # v3 table: top-500 scope, cell 12b
os.environ.setdefault("HEO_X4_SCOPE",     "top500")              # top500 | survivors | anchors | all  (PASS 3 scope)
os.environ.setdefault("HEO_MODE",         "orb")                 # orb | mace | mock
os.environ.setdefault("HEO_SQSCUTOFFS",   "6.0,4.0")             # v1-verified cluster cutoffs
os.environ.setdefault("HEO_SQSSTEPS",     "3000")
os.environ.setdefault("HEO_EHULLCUT",     "0.10")                # on Ehull_eff (entropy-corrected)
os.environ.setdefault("HEO_SAVE_RELAXED", "1")                   # every relaxed cell -> CIF, so no re-relax later
os.environ.setdefault("HEO_CHUNK",        "100")                 # compositions per SQS+relax cycle
os.environ.setdefault("HEO_TERMINATE_MODE", "none")              # stop | delete | none  (user 2026-09-03: never auto-terminate)

WORKDIR   = os.environ["HEO_WORKDIR"]
MODE      = os.environ["HEO_MODE"]
POD_ID, N_PODS = 0, 1                                  # only for splitting one run across several pods
LIMIT     = int(os.environ.get("HEO_LIMIT", 0))        # smoke-test cap on #compositions; 0 = all
MAX_HOURS = float(os.environ.get("HEO_MAX_HOURS", 36))
# 180 (was 60): the 2026-09-04 run lost the pod twice to the 60-min idle rule while a long CPU-only
# cell was running.  The watchdog is log-only anyway under HEO_TERMINATE_MODE=none.
IDLE_MIN  = float(os.environ.get("HEO_IDLE_MIN", 180))

# MP_API_KEY comes from the shell (export MP_API_KEY=...). Never write the key into this notebook:
# a notebook is shared and committed far more often than a shell session.
MP_API_KEY = os.environ.get("MP_API_KEY", "")

N_GPUS = 0
try:
    import torch; N_GPUS = torch.cuda.device_count()
except Exception:
    try:      # no torch yet (first run, before cell 2) -> ask the driver instead
        _o = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
        N_GPUS = sum(1 for l in _o.splitlines() if l.startswith("GPU "))
    except (FileNotFoundError, OSError):
        N_GPUS = 0                       # no driver either (local / mock box)
N_GPUS = int(os.environ.get("HEO_NGPU", 0)) or N_GPUS or 1

for sub in ("", "/structures/sqs", "/structures/relaxed", "/checkpoints", "/results", "/hull_cache"):
    os.makedirs(WORKDIR + sub, exist_ok=True)
Path(WORKDIR, "results", "DONE").unlink(missing_ok=True)   # a stale sentinel must not fire the watchdog

# ---- v4: the read-only base run ----
BASE_WORKDIR = os.environ.get("HEO_BASE_WORKDIR", "").rstrip("/")
if BASE_WORKDIR and not os.path.isdir(BASE_WORKDIR):
    print(f"[warn] HEO_BASE_WORKDIR={BASE_WORKDIR} not found -> no base cache: EVERY structure will be computed fresh")
    os.environ.pop("HEO_BASE_WORKDIR"); BASE_WORKDIR = ""
if BASE_WORKDIR:
    assert os.path.abspath(BASE_WORKDIR) != os.path.abspath(WORKDIR), "HEO_BASE_WORKDIR must differ from HEO_WORKDIR (base is read-only)"
    _bck = glob.glob(f"{BASE_WORKDIR}/checkpoints/energies*.csv")
    _nrow = sum(len(pd.read_csv(f)) for f in _bck)
    _ncif = len(os.listdir(f"{BASE_WORKDIR}/structures/relaxed")) if os.path.isdir(f"{BASE_WORKDIR}/structures/relaxed") else 0
    print(f"base run: {BASE_WORKDIR} | {len(_bck)} checkpoint files, {_nrow} relaxed rows, {_ncif} CIFs (read-only)")
    if _ncif == 0: print("  [warn] no relaxed CIFs in the base -> cell matrices of v3 structures unavailable (dV decomposition NaN for them)")
BASE_RESULTS = os.environ.get("HEO_BASE_RESULTS", "")
if BASE_RESULTS and not os.path.exists(BASE_RESULTS):
    print(f"[warn] HEO_BASE_RESULTS={BASE_RESULTS} not found -> HEO_X4_SCOPE=top500 cannot rank; use 'survivors'")

# ---- every notebook print also goes to results/notebook_stdout.log ----
# A browser "Run All" only keeps cell outputs while the tab stays connected; the kernel keeps running
# after a disconnect but those prints are lost from the cells.  The log keeps them all (and the
# papermill route in the last cell keeps them inside the cells as well).
class _Tee:
    _heo_tee = True
    def __init__(self, stream, path): self._s = stream; self._f = open(path, "a", buffering=1)
    def write(self, x): self._s.write(x); self._f.write(x); return len(x)
    def flush(self): self._s.flush(); self._f.flush()
    def __getattr__(self, a): return getattr(self._s, a)
_LOG = f"{WORKDIR}/results/notebook_stdout.log"
if not getattr(sys.stdout, "_heo_tee", False):
    sys.stdout = _Tee(sys.stdout, _LOG); sys.stderr = _Tee(sys.stderr, _LOG)
print(f"\n===== notebook run started {time.strftime('%Y-%m-%d %H:%M:%S')} -> also logging to {_LOG} =====")
print(f"MODE={MODE} | GPUs={N_GPUS} | workdir={WORKDIR} | LIMIT={LIMIT or 'all'} | x4 scope={os.environ['HEO_X4_SCOPE']}")
print(f"MP_API_KEY: {'set' if MP_API_KEY else 'NOT SET -> cell 8 will fail unless the hull cache exists'}")
print(f"terminate={os.environ['HEO_TERMINATE_MODE']} | max {MAX_HOURS} h | idle {IDLE_MIN} min")
if MODE != "mock":
    for i in range(N_GPUS):
        try: print(" ", torch.cuda.get_device_name(i))
        except Exception: pass

ModuleNotFoundError: No module named 'pandas'

## 셀 1b — 팟 과금 방지 워치독

노트북 셀은 **첫 예외에서 멈춥니다.** 앵커 게이트나 어떤 assert가 걸리면 셀 13에 도달하지 못하고, 커널은 살아 있으니 팟이 계속 과금됩니다.
이 셀은 커널과 무관하게 사는 프로세스를 띄워 ① 하드 데드라인 ② 파일 무활동 + GPU 유휴 ③ `DONE` 이후에도 팟이 살아 있음, 세 경우에 팟을 내립니다.

v1의 "워커 PID 전원 종료 → 삭제"는 **워커가 행(hang)하면 영원히 안 걸리고**, 반대로 PASS 1이 정상 종료한 직후에도 걸려서 hull 셀 전에 팟을 죽입니다.

In [ ]:
# ============ CELL 1b: pod cost guard — detached watchdog ============
# A notebook cell stops at the FIRST exception, so cell 13 is NOT reached when any assert fires
# (anchor gate, hull, invariants). The kernel then sits idle and the pod bills forever. This starts
# an independent process that outlives the kernel and terminates the pod on any of:
#   (1) hard deadline, (2) no file activity for IDLE_MIN while the GPUs are idle,
#   (3) results/DONE written by cell 13 but the pod still alive 5 min later.
# v1 terminated on "all worker PIDs exited", which never fires if a worker hangs and fires too
# early if it exits between passes (killing the pod before the hull cell). Set HEO_WATCHDOG=0 to disable.
import textwrap
WATCHDOG_SRC = textwrap.dedent(r"""
    import os, sys, time, subprocess, urllib.request
    from pathlib import Path
    WORKDIR = Path(sys.argv[1]); MAX_H = float(sys.argv[2]); IDLE_MIN = float(sys.argv[3]); MODE = sys.argv[4]
    LOG = open(WORKDIR / "watchdog.log", "a", buffering=1)
    def say(m): LOG.write(f"[{time.strftime('%H:%M:%S')}] {m}\n")

    # runpodctl first, REST second. Never raises: a missing binary must not skip the fallback.
    def pod_terminate(mode):
        pod = os.environ.get("RUNPOD_POD_ID")
        if not pod: return False, "RUNPOD_POD_ID not set"
        cmds = ([["runpodctl","pod","delete",pod],["runpodctl","remove","pod",pod]] if mode == "delete"
                else [["runpodctl","pod","stop",pod],["runpodctl","stop","pod",pod]])
        for c in cmds:
            try:
                if subprocess.run(c, capture_output=True).returncode == 0: return True, " ".join(c)
            except Exception as e: say(f"  {c[0]} unusable: {e!r}")
        key = os.environ.get("RUNPOD_API_KEY")
        if not key: return False, "runpodctl failed and RUNPOD_API_KEY not set"
        url = f"https://rest.runpod.io/v1/pods/{pod}" + ("" if mode == "delete" else "/stop")
        req = urllib.request.Request(url, method="DELETE" if mode == "delete" else "POST",
                                     headers={"Authorization": f"Bearer {key}"})
        try:
            urllib.request.urlopen(req, timeout=20); return True, "REST " + url
        except Exception as e: return False, f"REST failed: {e!r}"

    def gpus_busy():
        try:
            out = subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu","--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=30).stdout
            return any(int(x) > 5 for x in out.split() if x.strip().isdigit())
        except Exception:
            return False          # no nvidia-smi -> fall back to file activity alone

    def snapshot():
        s = []
        for p in WORKDIR.rglob("*"):
            if p.is_file() and p.name != "watchdog.log":
                try: s.append((str(p), p.stat().st_size, int(p.stat().st_mtime)))
                except OSError: pass
        return hash(tuple(sorted(s)))

    def fire(mode, why):
        if mode == "none":                       # terminate disabled by the user: log what WOULD happen
            say(f"LOG-ONLY - would terminate now: {why}")
            return
        say(f"TERMINATING ({mode}) - {why}")
        ok, how = pod_terminate(mode)
        say(f"  -> {'OK ' + how if ok else 'FAILED: ' + how}")
        sys.exit(0 if ok else 1)

    say(f"watchdog up | max {MAX_H} h | idle {IDLE_MIN} min | mode {MODE} | pod {os.environ.get('RUNPOD_POD_ID')}")
    t0 = time.time(); last_sig = snapshot(); last_change = time.time(); done_at = None
    while True:
        time.sleep(60)
        if (WORKDIR / "results" / "DONE").exists():
            done_at = done_at or time.time()
            if time.time() - done_at > 300:
                fire(MODE, "DONE written, pod still up after 5 min"); done_at += 21600   # re-log every 6 h in log-only
            continue
        if time.time() - t0 > MAX_H * 3600:
            fire(MODE, f"hard deadline {MAX_H} h"); t0 += 21600
        sig = snapshot()
        if sig != last_sig: last_sig, last_change = sig, time.time(); continue
        idle_s = time.time() - last_change
        if idle_s > IDLE_MIN * 60:
            if gpus_busy():
                say(f"  files idle {idle_s/60:.0f} min but GPUs busy - holding"); last_change = time.time()
            else:
                fire(MODE, f"no file activity for {idle_s/60:.0f} min and GPUs idle")
                last_change = time.time()        # log-only: re-arm instead of spamming every minute
""")

_wd_mode = os.environ["HEO_TERMINATE_MODE"]
# "none" now passes through unchanged: the watchdog still runs but only LOGS what it would have
# done (user decision 2026-09-03: this run must never be terminated automatically).
if os.environ.get("HEO_WATCHDOG", "1") == "1" and os.environ.get("RUNPOD_POD_ID"):
    _wd = Path(WORKDIR, "pod_watchdog.py"); _wd.write_text(WATCHDOG_SRC)
    _p = subprocess.Popen([sys.executable, str(_wd), WORKDIR, str(MAX_HOURS), str(IDLE_MIN), _wd_mode],
                          env=dict(os.environ), start_new_session=True,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"pod watchdog started (pid {_p.pid}) -> max {MAX_HOURS} h, idle {IDLE_MIN} min, "
          f"mode {_wd_mode}, log {WORKDIR}/watchdog.log")
    if _wd_mode == "none":
        print("  terminate DISABLED -> log-only watchdog; the pod BILLS until you stop it yourself")
elif not os.environ.get("RUNPOD_POD_ID"):
    print("pod watchdog: RUNPOD_POD_ID not set -> not on a RunPod pod, nothing to guard")
else:
    print("pod watchdog DISABLED (HEO_WATCHDOG=0)")

## 셀 2 — 패키지 설치

In [ ]:
# ============ CELL 2: INSTALL ============
# mace-torch is only needed for the contrast run (HEO_MODE=mace, separate HEO_WORKDIR).
# v4: import first, pip only for what is missing -- a fresh `pip install` on every Run All pulled
# unpinned torch-sim/orb-models versions and needed the network.  The versions actually loaded are
# written to results/pkg_versions.json and copied into audit.json by cell 11.
import importlib.util, subprocess, sys, json
from importlib.metadata import version as _ver, PackageNotFoundError
PKGS = {"pymatgen": "pymatgen", "mp_api": "mp-api", "icet": "icet", "ase": "ase",
        "torch_sim": "torch-sim-atomistic", "orb_models": "orb-models", "scipy": "scipy", "papermill": "papermill"}
need = list(PKGS) if MODE != "mock" else ["pymatgen", "scipy"]        # papermill: for the terminal route (last cell)        # mock needs no MLIP / icet
missing = [PKGS[m] for m in need if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
PKG_VERSIONS = {}
for mod, pip in PKGS.items():
    if importlib.util.find_spec(mod) is None: continue
    try: PKG_VERSIONS[pip] = _ver(pip)
    except PackageNotFoundError: PKG_VERSIONS[pip] = "?"
json.dump(PKG_VERSIONS, open(f"{WORKDIR}/results/pkg_versions.json", "w"), indent=1)
print("versions:", PKG_VERSIONS)
if MODE != "mock":
    import torch_sim, orb_models, icet
    print("torch-sim OK | orb-models OK | icet OK")

## 셀 3 — 워커 모듈 확인

`heo_worker.py`를 이 노트북과 **같은 디렉터리에 업로드**해 두세요. 임포트되는지, 템플릿 검증(적층 글자 + 프리즘 질서변수)이 통과하는지, 어떤 파일이 로드됐는지 md5로 확인합니다.

In [ ]:
# ============ CELL 3: worker module check ============
import hashlib, importlib, sys
sys.path.insert(0, os.getcwd())
import heo_worker as W; importlib.reload(W)
print("heo_worker.py:", W.__file__)
print("  md5:", hashlib.md5(open(W.__file__, "rb").read()).hexdigest())
print("  model tag:", W.MODEL_TAG, "| SQS cutoffs:", W.SQS_CUTOFFS, "| MC steps:", W.SQS_MC_STEPS)
print("  lattice a,c,z_O =", W.A_HEX, W.C_HEX, W.Z_O,
      "|", W.N_ATOMS, "atoms | remove", W.N_NA_REMOVE, "Na |", W.N_VAC_SAMPLES, "vacancy samples")
W._validate_templates(verbose=True)     # O3 CABCAB / P3 CAABBC, prismatic order -1 / +1, min Na-O > 2 A
# ---- v4 ----
print("  x grid (stem: Na removed / x):", {s: (n, round(W.X_OF[s], 4)) for s, n in W.X_STEMS.items()})
print("  vacancy salts:", W.VAC_SALT, "| Na allowed (invariant 8):", sorted(W.NA_ALLOWED), "+", W.NA_MILD, "mild only")
print("  base workdir:", W.BASE_WORKDIR or "(none)", "| ckpt fields:", W.CKPT_FIELDS)
assert (W.BASE_WORKDIR or "") == (BASE_WORKDIR or ""), "worker and notebook disagree on HEO_BASE_WORKDIR"
assert "cell" in W.CKPT_FIELDS and hasattr(W, "dV_report") and hasattr(W, "x_star_bracket"), "heo_worker.py is not the v4 worker"
W._check_na("TEMPLATE", "O3_pris", W.O3_TEMPLATE)          # invariant 8 self-test on the template (27 Na)
assert W.parse_tag("O3_x081_3") == ("O3", "x081", 3, 5) and W.parse_tag("P3_desod_0")[3] == W.N_NA_REMOVE
print("  v4 worker OK")

## 셀 4 — Stage 1: 조성 열거

27개 TM 자리의 **정확 정수 전하식** `Σ n_i·v_i = +81`. 다가 원소는 `DEFAULT_STATE`에서 출발하고 부족/과잉 전하는 호스트 redox 쌍이 먼저 흡수합니다.
채택된 배분(`n_Ni3`, `n_Mn3`, `n_Co2`, `n_V3`, `n_V5`)이 **DFT 자화 감사가 검증할 예측**입니다.

격자 A(Ni6Mn6 + 3-3-3-3-3) · B(Ni4Mn4 + 4-4-4-4-3) · C(구제 격자, A도 B도 정수해가 없는 조합에만).
`compositions_master.csv`는 **입력 전용**이라 이미 있으면 md5가 같은지만 확인하고 절대 덮어쓰지 않습니다.

In [ ]:
# ============ CELL 4: STAGE 1 — enumeration ============
import pandas as pd, hashlib, json
fp = f"{WORKDIR}/results/compositions_master.csv"
comps, n_legacy = W.enumerate_compositions(use_exact=True)
COMP_HASH = hashlib.md5(comps.to_csv(index=False).encode()).hexdigest()

if os.path.exists(fp):
    on_disk = hashlib.md5(open(fp, "rb").read()).hexdigest()
    assert on_disk == COMP_HASH, ("compositions_master.csv on disk differs from this enumeration "
                                  f"({on_disk} != {COMP_HASH}) -> delete it or restore the CONFIG")
    print("existing master reused, md5 identical:", len(comps), "rows")
else:
    tmp = fp + ".tmp"; comps.to_csv(tmp, index=False); os.replace(tmp, fp)   # atomic
    open(f"{WORKDIR}/results/compositions.md5", "w").write(COMP_HASH)
    print("master written:", len(comps), "rows")

print("exact-neutral:", len(comps), comps.scheme.value_counts().to_dict())
print("legacy-tolerance count (v1 reproduction, expect 7279):", n_legacy)
ent = {s: round(1000*W.K_B*W.T_SYNTH*comps[comps.scheme == s].sconf_per_atom_kB.mean(), 1) for s in "ABC"}
print("entropy bonus at T_synth, A/B/C (meV/atom):", ent)
print("compositions md5:", COMP_HASH)
# ---- invariant 3: the scheme-A entropy term must land in 42-52 meV/atom
assert 42 <= ent["A"] <= 52, f"entropy term {ent['A']} meV/atom out of range -> check the 27/108 factor or the log"
comps.head(3)

## 셀 5 — ★ 앵커 게이트 (재기 전에 자부터)

답을 아는 두 계로 자의 방향을 검증합니다 (2026-09-03 재설계, A안 = 상대 게이트):

- **HOST** NaNi₀.₅Mn₀.₅O₂ → 탈소듐 시 **P3로 gliding** (Komaba 2012). 13.5/13.5는 27자리에 못 앉으므로 두 변형 평균 — 1차 전하 오차가 상쇄됩니다.
- **HEO** NaNi₀.₁₂Cu₀.₁₂Mg₀.₁₂Fe₀.₁₅Co₀.₁₅Mn₀.₁Ti₀.₁Sn₀.₁Sb₀.₀₄O₂ (Zhao et al., *Angew. Chem.* **59**, 264 (2020)) → O3→P3 전이가 **지연**됨(용량의 60% 이상이 O3 영역). 반정수 문제가 없어 두 변형은 같은 조성에 다른 난수(SQS·빈자리) — 9원소 배열 노이즈를 평균합니다.

게이트는 **절대 상 판정이 아니라 자의 방향**을 요구합니다: `dE_desod(HEO) > dE_desod(HOST) + margin`.
x=0.52에서 HEO가 실험적으로 O3라는 보장이 없기 때문입니다(전이가 늦을 뿐 사라지지 않음). 스크리닝의 목표가
"HEO보다 더 버티는 조성"이므로, 앵커는 눈금 방향만 보정하고 억제 판정(`phase_desod==O3`)은 후보의 목표 정의로 남습니다.

> **Ti03 앵커 삭제(2026-09-03).** "Ti₀.₃이 x=0.52에서 O3 유지"는 출처가 없었고, 1차 문헌(Wang et al., *Adv. Mater.* **29**,
> 1700210 (2017))은 NaNi₀.₅Mn₀.₅₋ₓTiₓO₂ 전 계열에서 **가역적 O3−P3 전이**를 보고합니다. 실제 ORB 실행(2026-09-02)도
> Ti03 → P3 (dE_desod −31)를 냈는데, 이는 모델 오류가 아니라 문헌과 일치하는 답이었습니다.

전압 창은 [2.5, 3.4]로 완화 — GGA 계열 화폐는 층상 산화물 전압을 0.3–0.5 V 과소평가합니다(Aydinol PRB 1997). 실측 2.685 V는 실험 3.1 V 대비 정확히 그 오프셋.

앵커를 워커가 아니라 여기서 한 번만 돌리는 이유: 샤드마다 중복 계산하면 같은 `anchors.csv`와 같은 CIF에 동시에 쓰는 경쟁이 생깁니다.

`sigma_paired`와 `sigma_max`를 나란히 찍습니다. **`sigma_paired < sigma_max`이면 짝지은 빈자리가 실제로 오차를 상쇄하고 있다는 증거**입니다.
(mock에서는 빈자리 벌점이 O3/P3 독립 난수라 반대로 나옵니다 — 실전에서만 판별됩니다.)

**게이트 지점 이동(2026-09-03, 승인)**: Zhao 2020의 만충전이 x≈0.52(110 mAh/g ≈ Δx 0.48)라 그 지점에선 실험도 HEO가 P3 —
방향 게이트를 실험이 두 앵커를 실제로 구분하는 **x = 20/27 ≈ 0.74**(Na 7개 제거, `{ph}_mild_{k}` 태그)로 옮김.
x=0.52 gap은 기록만 하고 게이트하지 않음. 절대 상 판정은 어느 x에서도 요구하지 않음(방향형 유지 — 계통 오차 상쇄).

**게이트 v4 (2026-09-03, 승인)**: ORB(−4.9)·MACE(−5.2)가 x=0.74 gap을 1 meV 안쪽으로 일치 판독 → 탈소듐 순서는
0 K PBE 열역학에 없는 신호(전이 지연 = 엔트로피/운동학 기원, npj Comput. Mater. 2025)로 판정하고 **진단으로 강등**.
게이트는 화폐가 실제로 분해하는 **pristine 방향**(`dE_pris(HEO) − dE_pris(HOST) > margin`, ORB +20.0/MACE +23.4)으로 이동,
"자가 충실한가"의 검증은 다음 셀 5c(단일 TM 문헌 DFT 캘리브레이션)가 담당.


In [ ]:
# ============ CELL 5: ANCHOR GATE ============
# Gate history (all user-approved, 2026-09-03):
#   v1  absolute Ti03 anchor -- removed (no literature support; Wang Adv. Mater. 2017 contradicts).
#   v2  directional gate at x=0.52 -- retired: Zhao 2020 fully charges the HEO to x~0.52
#       (110 mAh/g of ~229 mAh/g per Na), so BOTH anchors are P3 there in experiment.
#   v3  directional gate at x=0.74 -- retired as a GATE after ORB (-4.9) and MACE (-5.2)
#       agreed within 1 meV/f.u. on a ~0 gap: two independent architectures on one training
#       currency read the same value, so the missing signal is not model noise -- the O3->P3
#       delay in the HEO is not contained in 0 K PBE-family thermodynamics (entropic/kinetic
#       origin; cf. the npj Comput. Mater. 2025 entropic-control study, doi:10.1038/s41524-025-01954-2).
#       gap_074/gap_052 stay RECORDED as diagnostics: they are paper content, not gates.
#   v4  (current) directional gate on the PRISTINE preference, where the currency does resolve
#       the two anchors (ORB +20.0, MACE +23.4 meV/f.u.) -- plus the single-TM literature-DFT
#       calibration in CELL 5c, which takes over the "is the ruler faithful?" role.
import numpy as np, pandas as pd, json, time
ckpt_main = f"{WORKDIR}/checkpoints/energies_main.csv"
done = W.load_done_all()
W.merged_energies()          # invariant 6: one currency in the checkpoints, checked BEFORE any GPU time

E_NA = W.na_reference(ckpt_main, done)
print("E(Na)/atom =", round(E_NA, 4), "eV   (PBE reference ~ -1.31)")
json.dump({"E_Na_per_atom": E_NA, "model": W.MODEL_TAG}, open(f"{WORKDIR}/results/E_Na.json", "w"))

t0 = time.time()
items = [it for aid, c in W.ANCHORS.items() for w in ("all", "mild") for it in W.build_items(aid, c, w)]
print(f"{len(items)} anchor structures built in {time.time()-t0:.1f}s  (incl. x=0.74 diagnostic point)")
W.relax_batch(items, ckpt_main, done, counts_of=W.ANCHORS, save_cif=True)

X_MILD = (W.N_NA - W.N_NA_REMOVE_MILD) / W.N_NA          # 20/27 = 0.741

def mild_stats(recs):
    """Paired minimum and spread at x=0.74, same conventions as metrics_from (section 3):
    min_k(E_P3,k - E_O3,k) in meV/f.u., O3 and P3 sharing each vacancy pattern k."""
    ks = [k for k in range(W.N_VAC_SAMPLES) if f"O3_mild_{k}" in recs and f"P3_mild_{k}" in recs]
    assert ks, "no mild-desodiation records -> worker missing the 'mild' build path?"
    d = [(recs[f"P3_mild_{k}"]["energy"] - recs[f"O3_mild_{k}"]["energy"]) * 1000 / W.N_FU for k in ks]
    return float(min(d)), float(np.std(d))

R = W.recs_by_comp(W.merged_energies(), list(W.ANCHORS))
rows = []
for aid in W.ANCHORS:
    m = W.metrics_from(R[aid]); m["anchor"] = aid; m["group"] = aid.split("_")[0]
    o3 = [R[aid][f"O3_desod_{k}"]["energy"] for k in range(W.N_VAC_SAMPLES)]
    p3 = [R[aid][f"P3_desod_{k}"]["energy"] for k in range(W.N_VAC_SAMPLES)]
    m["sigma_max"]      = max(np.std(o3), np.std(p3))*1000/W.N_FU   # the unpaired estimate, for comparison
    m["dE_desod_indep"] = (min(p3) - min(o3))*1000/W.N_FU           # the pre-pairing definition
    m["dE_mild"], m["sigma_mild"] = mild_stats(R[aid])
    rows.append(m)
A = pd.DataFrame(rows)
cols = ["V_avg","dE_pris","dE_desod","dE_mild","ddE","dV_O3","dV_trans",
        "sigma_vac","sigma_mild","sigma_max","dE_desod_indep"]
Ag = A.groupby("group")[cols].mean()
Ag["delta"]      = A.groupby("group").sigma_vac.max().clip(lower=W.DELTA_FLOOR_MEV_FU)
Ag["delta_mild"] = A.groupby("group").sigma_mild.max().clip(lower=W.DELTA_FLOOR_MEV_FU)
Ag["phase_desod"] = [W.judge_phase(r.dE_desod, r.delta) for r in Ag.itertuples()]
Ag["phase_mild"]  = [W.judge_phase(r.dE_mild, r.delta_mild) for r in Ag.itertuples()]
Ag["variant_sign_agree"] = A.groupby("group").dE_pris.apply(lambda s: len(set(np.sign(s))) == 1)
A.to_csv(f"{WORKDIR}/results/anchors_raw.csv", index=False)
Ag.to_csv(f"{WORKDIR}/results/anchors.csv")
print(Ag.round(3).to_string())
print("\npaired-vacancy check:  sigma_paired", Ag.sigma_vac.round(2).to_dict(),
      "vs sigma_max", Ag.sigma_max.round(2).to_dict())
print("  paired < max -> the shared vacancy pattern is cancelling, as intended.")
print("  paired > max -> O3 and P3 vacancy energies are uncorrelated; pairing buys nothing, tell Claude.")

# ---- desodiated orderings: DIAGNOSTICS ONLY (kept for the paper, never gated) ----
gap_074 = float(Ag.loc["HEO", "dE_mild"] - Ag.loc["HOST", "dE_mild"])
gap_052 = float(Ag.loc["HEO", "dE_desod"] - Ag.loc["HOST", "dE_desod"])
print(f"\n[diagnostic] gap @ x={X_MILD:.2f} = {gap_074:+.1f}  |  gap @ x=0.52 = {gap_052:+.1f} meV/f.u.")
print("  ~0 or negative here is the EXPECTED stock-uMLIP reading (ORB -4.9 / MACE -5.2 on"
      "\n  2026-09-03): the experimental O3->P3 delay is not a 0 K-thermodynamic signal.")

# ---- invariant 5 (v4): gate on what 0 K thermodynamics does support ----
# (a) sanity: the HOST voltage over x=1 -> 0.52.  [2.5, 3.4] allows the systematic 0.3-0.5 V
#     GGA-currency underestimate (Aydinol PRB 1997); experiment ~3.1 V, real ORB 2.69 V.
assert 2.5 <= Ag.loc["HOST", "V_avg"] <= 3.4, f"HOST voltage {Ag.loc['HOST','V_avg']:.2f} V outside [2.5, 3.4]"
# (b) the negative anchor: HOST must glide at full charge, as in experiment (Komaba 2012).
assert Ag.loc["HOST", "phase_desod"] == "P3", "HOST must glide (P3) at x=0.52 -> model or template problem"
# (c) the pristine DIRECTION: at x=1 the currency resolves the chemistry (both uMLIPs read
#     HEO's O3 preference ~20 meV/f.u. above HOST's).  margin = the variant spread of the
#     2-variant mean (std over 2 values = half their difference), floored at 5 meV/f.u.
gap_pris    = float(Ag.loc["HEO", "dE_pris"] - Ag.loc["HOST", "dE_pris"])
margin_pris = float(max(A.groupby("group").dE_pris.std(ddof=0).max(), W.DELTA_FLOOR_MEV_FU))
print(f"\nruler direction @ x=1 (pristine): dE_pris(HEO) - dE_pris(HOST) = {gap_pris:+.1f} meV/f.u."
      f"  (margin {margin_pris:.1f})")
assert gap_pris > margin_pris, \
    f"pristine ruler direction wrong or lost in noise: gap {gap_pris:+.1f} <= margin {margin_pris:.1f} meV/f.u."

RULER = dict(dE_desod_HOST=float(Ag.loc["HOST", "dE_desod"]), dE_desod_HEO=float(Ag.loc["HEO", "dE_desod"]),
             ddE_HOST=float(Ag.loc["HOST", "ddE"]), ddE_HEO=float(Ag.loc["HEO", "ddE"]),
             dE_pris_HOST=float(Ag.loc["HOST", "dE_pris"]), dE_pris_HEO=float(Ag.loc["HEO", "dE_pris"]),
             dE_mild_HOST=float(Ag.loc["HOST", "dE_mild"]), dE_mild_HEO=float(Ag.loc["HEO", "dE_mild"]),
             x_mild=X_MILD, gap=gap_pris, margin=margin_pris, gap_074=gap_074, gap_052=gap_052,
             E_Na=float(E_NA), model=W.MODEL_TAG)
json.dump(RULER, open(f"{WORKDIR}/results/ruler.json", "w"), indent=1)
print("\nRULER (meV/f.u.):", {k: round(v, 1) for k, v in RULER.items() if isinstance(v, float)})
if gap_052 <= 0:
    print("[note] gap @ x=0.52 <= 0: cell 11's ruler_pos (built on the x=0.52 anchors) has an inverted"
          "\n       denominator -- read it as a coordinate, not as 'closer to 1 = better'.")
print("ANCHOR GATE PASSED -> run the CELL 5c calibration next")


## 셀 5c — ★ 자 캘리브레이션 (단일 TM, 문헌 0 K DFT 대조)

스크리닝이 의존하는 사슬은 `MLIP →(1)→ PBE 화폐 →(2)→ 실험` 중 **(1)뿐**입니다(같은 화폐 안에서의 랭킹).
(2)의 탈소듐 구간은 엔트로피/운동학이라 top-30 DFT·유한온도 후속 연구 몫. 이 셀은 (1)을 **발표된 0 K DFT가 있는
단순계**로 검증합니다: NaCoO₂/NaNiO₂/NaMnO₂ (27×3가 = +81 정확, SQS 불필요, 몇 분 계산).
assert는 확실한 문헌 사실만(부호·단조성·Co의 x=0.52 P3): Toumar PR Applied 2015, Kaufman & Van der Ven PRM 2019,
Delmas 1981 (O3→O′3→P3). 표의 정량값(기울기/연화)은 논문 그림과 눈으로 대조 — 실패 시 스크리닝 중단.

In [ ]:
# ============ CELL 5c: RULER CALIBRATION — single-TM set vs literature 0 K DFT ============
# The screening only needs link (1) of the chain  MLIP --(1)--> PBE-family DFT --(2)--> experiment:
# it ranks 7,073 compositions inside ONE currency.  Link (2) at desodiated x is entropic/kinetic
# (cell 5 diagnostics) and belongs to the top-30 DFT + finite-T follow-up.  This cell certifies
# link (1) on chemistries where the 0 K DFT answer is PUBLISHED, so no new DFT is needed:
#   NaCoO2 / NaNiO2 / NaMnO2 -- 27 x M(3+) = +81 satisfies charge neutrality exactly, single
#   element needs no SQS, and the O3/P3 x-dependence is established GGA(+U) literature
#   (Toumar et al. Phys. Rev. Applied 4, 064002 (2015): O3-type NaxMO2 voltages/orderings;
#    Kaufman & Van der Ven PRM 3, 015402 (2019): NaxCoO2 stacking hulls; experimental sequence
#    O3 -> O'3 -> P3 on charge: Delmas et al. 1981).
# Asserted here are only the ROBUST literature facts (signs and orderings); the printed table is
# for quantitative comparison against the papers' figures (slope/softening), not asserted.
import numpy as np, pandas as pd, json

CAL = {"CAL_Co": {"Co": 27}, "CAL_Ni": {"Ni": 27}, "CAL_Mn": {"Mn": 27}}
assert np.isfinite(W.E_NA_GLOBAL[0]), "E_Na not set -> run cell 5 first"

def cal_items(cid, counts):
    o3 = W.place_tm_random(counts, W.comp_rng(cid))       # single element: placement is trivial
    p3 = W.o3_to_p3(o3, glide=W.GLIDE)
    yield cid, "O3_pris", o3
    yield cid, "P3_pris", p3
    for salt0, nrm, name in ((100, W.N_NA_REMOVE, "desod"), (200, W.N_NA_REMOVE_MILD, "mild")):
        for k in range(W.N_VAC_SAMPLES):
            pat = W.vacancy_pattern(o3, W.comp_rng(cid, salt=salt0+k), n_remove=nrm)
            for ph, st in (("O3", o3), ("P3", p3)):
                d = st.copy(); d.remove_sites(pat)
                yield cid, f"{ph}_{name}_{k}", d

items = [it for cid, c in CAL.items() for it in cal_items(cid, c)]
print(f"{len(items)} calibration structures (3 compounds x [2 pris + 2x{W.N_VAC_SAMPLES} desod"
      f" + 2x{W.N_VAC_SAMPLES} mild])")
W.relax_batch(items, ckpt_main, done, counts_of=CAL, save_cif=True)

def paired_min(recs, name):
    ks = [k for k in range(W.N_VAC_SAMPLES) if f"O3_{name}_{k}" in recs and f"P3_{name}_{k}" in recs]
    d  = [(recs[f"P3_{name}_{k}"]["energy"] - recs[f"O3_{name}_{k}"]["energy"]) * 1000 / W.N_FU for k in ks]
    return float(min(d)), float(np.std(d))

Rc = W.recs_by_comp(W.merged_energies(), list(CAL))
crow = []
for cid in CAL:
    m = W.metrics_from(Rc[cid])                                   # dE_pris, dE_desod, V_avg (x=0.52 window)
    m["compound"] = cid.replace("CAL_", "Na") + "O2"
    m["dE_mild"], m["sigma_mild"] = paired_min(Rc[cid], "mild")
    # crossing x where dE(P3-O3) = 0, linear between the bracketing pair of the 3 computed x points
    xs = [1.0, (W.N_NA - W.N_NA_REMOVE_MILD)/W.N_NA, (W.N_NA - W.N_NA_REMOVE)/W.N_NA]
    ds = [m["dE_pris"], m["dE_mild"], m["dE_desod"]]
    m["x_cross"] = float("nan")
    for (x1, d1), (x2, d2) in zip(zip(xs, ds), list(zip(xs, ds))[1:]):
        if d1 > 0 >= d2:
            m["x_cross"] = x1 + (x2 - x1) * d1 / (d1 - d2); break
    crow.append(m)
C = pd.DataFrame(crow).set_index("compound")
show = ["V_avg", "dE_pris", "dE_mild", "dE_desod", "sigma_vac", "sigma_mild", "x_cross"]
print(C[show].round(3).to_string())
C.to_csv(f"{WORKDIR}/results/calibration.csv")
json.dump({"model": W.MODEL_TAG, "table": C[show].round(3).to_dict()},
          open(f"{WORKDIR}/results/calibration.json", "w"), indent=1, default=str)

# ---- calibration invariants: the ruler must reproduce the published 0 K facts ----
# (1) at x=1 every NaMO2 prefers O-type stacking (synthesis + every published hull)
assert (C.dE_pris > 0).all(), f"calibration: O3 not preferred at x=1 -> broken ruler\n{C.dE_pris}"
# (2) desodiation must monotonically favor P-type: dE_pris > dE_mild > dE_desod
assert ((C.dE_pris > C.dE_mild) & (C.dE_mild > C.dE_desod)).all(), \
    f"calibration: dE(x) not monotonic -> broken ruler\n{C[['dE_pris','dE_mild','dE_desod']]}"
# (3) the cleanest case, NaxCoO2: P-type is favored by mid-desodiation (O3->O'3->P3, Delmas 1981;
#     the published stacking hulls agree) -> dE at x=0.52 must be negative
assert C.loc["NaCoO2", "dE_desod"] < 0, \
    f"calibration: NaxCoO2 not P3 at x=0.52 (dE={C.loc['NaCoO2','dE_desod']:+.1f}) -> broken ruler"
# (4) unit sanity (invariant 2 applied to the calibration set)
assert C[["dE_pris", "dE_mild", "dE_desod"]].abs().max().max() < 500, "calibration: unit slip"
# soft check, not asserted: GGA-family average voltages of these compounds sit around 2-3.3 V
for cmpd, v in C.V_avg.items():
    if not (1.8 <= v <= 3.6):
        print(f"  [warn] {cmpd} V_avg {v:.2f} V outside the loose GGA window [1.8, 3.6] -- compare"
              f" against Toumar 2015 before trusting the run")
print("\nCALIBRATION PASSED -> the ruler reproduces the published 0 K stacking facts;"
      "\ncompare the table against Toumar 2015 / Kaufman 2019 figures for the slope, then launch PASS 1")


## 셀 5d — ★ 앵커 x4 파일럿 (HOST / HEO, 4-point Na 격자)

스펙 §5-2의 파일럿. HOST·HEO 두 앵커(각 2변형)에 x = 22/27, 18/27 두 점을 추가해 4점 dE(x) → verdict(x) → x\* 구간을 잰다.

**중단 조건(2026-09-10 사용자 승인, 스펙 기대치의 완화판)**: (i) HOST가 창 안에서 `transition` 또는 `amb_boundary`,
(ii) 두 앵커 모두 `nonmonotonic` 아님, (iii) 모든 V_seg > 0(불변량 9). **HEO−HOST 순서는 진단 기록만** — 이 화폐(0 K PBE)에
HEO 전이 지연 신호가 없다는 것은 heo_v3.md·셀11결과.md에서 이미 확정(ORB·MACE 일치, x_cross HOST 0.570 < HEO 0.607)이므로
4점에서도 둘이 같은 구간 `transition (0.52, 0.67)`로 나오는 것이 **기대되는** 결과다. Ti0.3 앵커는 2026-09-03에 삭제됨.

`HEO_X4_SCOPE=anchors`이면 여기까지가 v4 계산의 전부이고 PASS 3 셀들은 no-op이다.

In [ ]:
# ============ CELL 5d: ANCHOR x4 PILOT — 4-point Na grid on HOST / HEO (CHANGE_SPEC_v4 section 5-2) ============
# Same machine as cell 5 (build_items -> relax_batch -> metrics), extended to the two interior x points.
# Stop conditions are the relaxed ones approved 2026-09-10 (see the markdown above); the HEO-vs-HOST
# ordering is a diagnostic because the 0 K currency is already known not to contain the delay signal.
import numpy as np, pandas as pd, json, time
X4_SCOPE = os.environ.get("HEO_X4_SCOPE", "top500")
assert np.isfinite(W.E_NA_GLOBAL[0]), "E_Na not set -> run cell 5 first"
E_NA = W.E_NA_GLOBAL[0]; floor = W.DELTA_FLOOR_MEV_FU

t0 = time.time()
items = [it for aid, c in W.ANCHORS.items() for it in W.build_items(aid, c, "x4")]
W.relax_batch(items, ckpt_main, done, counts_of=W.ANCHORS, save_cif=True)
print(f"{len(items)} anchor x4 structures (4 anchors x 2 x-points x 2 phases x {W.N_VAC_SAMPLES} samples) in {time.time()-t0:.1f}s")

R = W.recs_by_comp(W.merged_energies(), list(W.ANCHORS))
rows = []
for aid in W.ANCHORS:
    rec = R[aid]; m = W.metrics_from(rec); m["anchor"] = aid; m["group"] = aid.split("_")[0]
    for s in W.X4_STEMS:
        mx = W.metrics_x(rec, s); assert mx, f"{aid}: no complete {s} pair"; m.update(mx)
    rows.append(m)
A4 = pd.DataFrame(rows).set_index("anchor")
A4.to_csv(f"{WORKDIR}/results/anchors_x4_raw.csv")

# group level: mean dE per x (as cell 5), delta(x) = max over variants of sigma_vac(x), floored
stems = W.X_GRID_DESC                                   # pris, x081, x067, desod  (x descending)
dE_col = {"pris": "dE_pris", "x081": "dE_x081", "x067": "dE_x067", "desod": "dE_desod"}
sg_col = {"x081": "sigma_vac_x081", "x067": "sigma_vac_x067", "desod": "sigma_vac"}
G = {}
for grp, sub in A4.groupby("group"):
    g = {"group": grp}
    verdicts, E_nat, src = [], {}, {}
    for s in stems:
        dE = float(sub[dE_col[s]].mean()); g[dE_col[s]] = dE
        if s == "pris":
            v = "O3" if dE > floor else W.judge_phase(dE, floor); g["delta_pris"] = floor
            E_nat[s], src[s] = float(sub.E_pris_O3.mean()), "O3"
        else:
            delta = float(max(sub[sg_col[s]].max(), floor)); g[f"delta_{s}"] = delta
            v = W.judge_phase(dE, delta)
            eo = float(sub[("E_desod_O3" if s == "desod" else f"E_O3_{s}")].mean())
            ep = float(sub[("E_desod_P3" if s == "desod" else f"E_P3_{s}")].mean())
            E_nat[s], src[s] = W.e_nat_of(v, eo, ep)
        g[f"verdict_{s}"] = v; verdicts.append(v)
    g.update(W.x_star_bracket(verdicts)); g.update(W.v_segments(E_nat, E_NA)); g.update(W.formation_x(E_nat))
    g["E_nat_src"] = "/".join(src[s] for s in stems)
    G[grp] = g
X4 = pd.DataFrame(G.values()).set_index("group")

# change A on the anchors: exact-cell decomposition per x (variant 1 of each group, k_best of each x)
dv_rows = []
for aid in W.ANCHORS:
    rec = R[aid]; cp = W.cell_of(aid, "O3_pris", rec["O3_pris"])
    for s in ("x081", "x067", "desod"):
        kb = int(A4.loc[aid, "k_best" if s == "desod" else f"k_best_{s}"])
        v = X4.loc[aid.split("_")[0], f"verdict_{s}"]; ph = "O3" if v == "O3" else "P3"
        cd = W.cell_of(aid, f"{ph}_{s}_{kb}", rec.get(f"{ph}_{s}_{kb}"))
        if cp is None or cd is None: continue
        r = W.dV_report(cp, cd); r.update(anchor=aid, x=round(W.X_OF[s], 4), phase=ph)
        r["ident_resid_pp"] = 100 * ((1 + r["dV_pct"]/100) - (1 + r["dA_pct"]/100) * (1 + r["dh_perp_pct"]/100))
        dv_rows.append(r)
DV = pd.DataFrame(dv_rows)
X4.to_csv(f"{WORKDIR}/results/anchors_x4.csv"); DV.to_csv(f"{WORKDIR}/results/anchors_x4_dV.csv", index=False)
json.dump({"model": W.MODEL_TAG, "groups": {k: {kk: (vv if not isinstance(vv, float) else round(vv, 4)) for kk, vv in v.items()}
                                              for k, v in G.items()}}, open(f"{WORKDIR}/results/anchors_x4.json", "w"), indent=1, default=str)

show = ["dE_pris", "dE_x081", "dE_x067", "dE_desod", "verdict_pris", "verdict_x081", "verdict_x067", "verdict_desod",
        "x_star_class", "x_star_lo", "x_star_hi", "x_star_mid", "V_seg_100_081", "V_seg_081_067", "V_seg_067_052",
        "Ef_x081", "Ef_x067", "convexity_flag"]
print("\nanchor 4-point table (dE meV/f.u., V_seg V, Ef meV/f.u.):")
print(X4[show].T.to_string())
print("\nchange A on the anchors (exact cells, k_best of each x):")
print(DV[["anchor", "x", "phase", "dV_pct", "dA_pct", "dh_perp_pct", "vm_strain", "d_inter_pris", "ident_resid_pp"]].round(4).to_string(index=False))

# ---- diagnostics (recorded, never gated) ----
print(f"\n[diagnostic] x* bracket: HOST {X4.loc['HOST','x_star_class']} ({X4.loc['HOST','x_star_lo']:.3f}, {X4.loc['HOST','x_star_hi']:.3f})"
      f" | HEO {X4.loc['HEO','x_star_class']} ({X4.loc['HEO','x_star_lo']:.3f}, {X4.loc['HEO','x_star_hi']:.3f})")
print("  gap(HEO-HOST) per x, meV/f.u.:", {s: round(float(X4.loc['HEO', dE_col[s]] - X4.loc['HOST', dE_col[s]]), 1) for s in stems})
print("  the same bracket for both is the EXPECTED 0 K reading (heo_v3.md section 3; x_cross HOST 0.570 < HEO 0.607).")
if len(DV) and DV.ident_resid_pp.abs().max() > 0.1:
    print(f"  [WARN] dV identity residual up to {DV.ident_resid_pp.abs().max():.3f} %p -> check the cell pipeline")

# ---- stop conditions (user-approved relaxation of spec section 5-2, 2026-09-10) ----
assert X4.loc["HOST", "x_star_class"] in ("transition", "amb_boundary"), \
    f"anchor pilot: HOST must glide inside the window (got {X4.loc['HOST','x_star_class']}) -> model/template problem"
assert not (X4.x_star_class == "nonmonotonic").any(), f"anchor pilot: non-monotonic verdicts {X4.x_star_class.to_dict()} -> review vacancy sampling"
_vs = X4[[c for c in X4.columns if c.startswith("V_seg_")]]
assert (_vs > 0).all().all(), f"invariant 9: a segment voltage is not positive\n{_vs}"
print("\nANCHOR x4 PILOT PASSED", "-> HEO_X4_SCOPE=anchors: PASS 3 cells below are no-ops" if X4_SCOPE == "anchors" else "-> continue")

## 셀 6 — PASS 1 런처 (pristine O3/P3 전수)

GPU당 워커 하나, 각자 자기 체크포인트 CSV에만 씁니다(경쟁 없음). 이 셀은 **즉시 반환**하니 셀 7로 지켜보세요.
워커는 100조성씩 끊어 **SQS를 전체 CPU 코어로 먼저 생성한 뒤** GPU 배치로 넘깁니다 — GPU가 SQS를 기다리며 놀지 않게 하는 핵심입니다.

In [ ]:
# ============ CELL 6: PASS 1 LAUNCHER ============
import subprocess, sys, os
PROCS = {}
def launch(pass_no):
    total = N_PODS * N_GPUS; procs = []
    for k in range(N_GPUS):
        wid = POD_ID * N_GPUS + k
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
        log = open(f"{WORKDIR}/checkpoints/worker_{wid}_p{pass_no}.log", "a")
        cmd = [sys.executable, "heo_worker.py", "--worker-id", str(wid),
               "--n-workers", str(total), "--pass-no", str(pass_no)]
        if LIMIT: cmd += ["--limit", str(LIMIT)]
        p = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
        procs.append(p); print(f"worker {wid} -> GPU {k} (pid {p.pid})")
    PROCS[pass_no] = procs
    print(f"\n{len(procs)} workers running (PASS {pass_no}). This cell returns now - watch with the next cell.")
    return procs

launch(1)

## 셀 7 — 모니터 (PASS 1이 끝날 때까지 반복 실행)

In [ ]:
# ============ CELL 7: MONITOR ============
# Re-run this cell to refresh, or call monitor(n, wait=True) to block until the pass finishes.
# v4: with wait=True a non-zero worker exit relaunches the pass ONCE (checkpoints make it resume from
# where it stopped); only a second failure raises.  The 2026-09-04 run saw a shard die on a transient
# Warp CUDA error, which under Run All used to stop the whole notebook at the barrier.
import glob, time, subprocess
import pandas as pd

def _gpu_line():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                               "--format=csv,noheader"], capture_output=True, text=True).stdout
    except (FileNotFoundError, OSError):
        return "(no nvidia-smi)\n"

def monitor(pass_no, wait=False, every=120, relaunch_once=True):
    relaunched = False
    while True:
        for lf in sorted(glob.glob(f"{WORKDIR}/checkpoints/worker_*_p{pass_no}.log")):
            lines = [l for l in open(lf).read().strip().split("\n") if l.strip()]
            print(f"--- {lf.split('/')[-1]} ---"); print("\n".join(lines[-3:]) or "  (no output yet)")
        procs = PROCS.get(pass_no, [])
        codes = [p.poll() for p in procs]; alive = sum(c is None for c in codes)
        print(f"\nalive: {alive}/{len(procs)} | exit codes: {codes}")
        fps = glob.glob(f"{WORKDIR}/checkpoints/energies*.csv")
        if fps: print("relaxed structures on disk (this workdir):", sum(len(pd.read_csv(f)) for f in fps))
        print(_gpu_line())
        if procs and alive == 0:
            if not all(c == 0 for c in codes):
                # a non-zero exit means the shard stopped early; its compositions are simply missing
                # downstream, so surface it here instead of letting cell 8 aggregate a partial pass.
                if wait and relaunch_once and not relaunched:
                    relaunched = True
                    print(f"[barrier] worker(s) exited non-zero {codes} -> relaunching PASS {pass_no} once (checkpoints resume)")
                    launch(pass_no); time.sleep(30); print("\n" + "-"*60); continue
                assert False, f"a worker exited non-zero {codes} -> read its log, fix, re-run the launcher (checkpoints resume)"
            print(f"PASS {pass_no} COMPLETE -> continue to the next cell")
            return
        if not wait: return
        time.sleep(every); print("\n" + "-"*60)

monitor(1)

## 셀 7b — ★ 배리어: PASS 1 완료까지 대기 (Run All 안전장치)

In [ ]:
# ============ CELL 7b: BARRIER — block Run All until PASS 1 is complete ============
# launch() returns immediately (workers are background subprocesses), so under "Run All" the
# cells below would aggregate a HALF-FILLED checkpoint.  This cell blocks: it re-prints worker
# logs every 2 min, returns only when every worker exited 0 (one automatic relaunch on a crash),
# and raises otherwise.  Restored 2026-09-10: this code cell was missing from the pod copy, which is
# why a Run All stopped at cell 8's coverage assert right after the PASS 1 launcher.
# (If the kernel was restarted after launch(), PROCS is empty and this cannot see the workers:
#  re-run the launcher -- checkpoints make it resume -- or rely on the cell-8 coverage assert.)
assert PROCS.get(1), "no PASS 1 workers tracked -> run the cell-6 launcher first"
monitor(1, wait=True)

## 셀 8 — Stage 2: hull (축 ①)

MP에서 **조성 원소 풀의 부분집합인 안정상**을 한 번에 받아 **같은 모델로 재이완**한 뒤 `pymatgen.PhaseDiagram`으로 `Ehull`을 구합니다.

v1이 여기서 한 번 죽었습니다 — 1원자짜리 원소 셀이 배치에 섞이니 autobatcher가 배치를 25만 원자까지 키우려다 폭주했습니다.
그래서 **24원자 미만은 슈퍼셀로 확장하고 크기순으로 정렬해** 배치를 균질하게 만듭니다(v1 CELL 7-FIX). 확장한 만큼 조성도 `factor`로 스케일합니다.

`Ehull_eff = Ehull − k_B·T·S_conf·(27/108)`, `T* = Ehull / (k_B·S_conf·27/108)`.

In [ ]:
# ============ CELL 8: HULL + PASS-1 AGGREGATION ============
import math, json, time, os
import numpy as np, pandas as pd
from pathlib import Path
from scipy.optimize import linprog
from pymatgen.core import Structure, Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry

CACHE = f"{WORKDIR}/hull_cache/competitors.json"
# v4: the competitor set is model-tagged and identical for the same MODEL_TAG -> take the base run's copy
# (a few MB) instead of re-querying MP; the base file itself is never modified.
import shutil
_base_cache = f"{BASE_WORKDIR}/hull_cache/competitors.json" if BASE_WORKDIR else ""
if not os.path.exists(CACHE) and _base_cache and os.path.exists(_base_cache):
    shutil.copy(_base_cache, CACHE); print("hull cache copied from the base run:", _base_cache)
ckpt_main = f"{WORKDIR}/checkpoints/energies_main.csv"
HEARTBEAT = Path(WORKDIR, "results", ".heartbeat")   # touched inside long CPU loops so the idle watchdog sees activity
done = W.load_done_all()
comps = pd.read_csv(f"{WORKDIR}/results/compositions_master.csv")
if LIMIT: comps = comps.sample(LIMIT, random_state=W.SEED).sort_values("comp_id").reset_index(drop=True)

# ---- mock: no MP, no relaxation; the synthetic Ehull keeps the downstream chain testable ----
MOCK_HULL = (MODE == "mock")
if MOCK_HULL: print("MODE=mock -> synthetic Ehull, MP is not queried")

# ---- competing phases: stable entries whose elements are a subset of our pool ----
# Section-6 item 4, verified 2026-09-03: exclude_elements is capped at 60 CHARACTERS server-side
# (HTTP 422), so one big exclusion query cannot work.  Inverted two-step instead:
#   (1) one LIGHT query listing every stable entry (id + element list only, ~40k rows, no structures),
#   (2) filter locally to element sets inside our pool, (3) fetch structures for just those ids in chunks.
if not MOCK_HULL and not os.path.exists(CACHE):
    assert MP_API_KEY, "export MP_API_KEY before running this cell"
    from mp_api.client import MPRester
    pool = sorted({e for sc in comps.site_counts for e in json.loads(sc)} | {"Na", "O"})
    pset = set(pool)
    with MPRester(MP_API_KEY) as mpr:
        light = mpr.materials.summary.search(is_stable=True, num_elements=(1, 7),
                                             fields=["material_id", "elements"])
        keep = [str(d.material_id) for d in light if {str(el) for el in d.elements} <= pset]
        print(f"stable MP entries: {len(light)} -> inside our {len(pool)}-element pool: {len(keep)}")
        raw = {}
        for i0 in range(0, len(keep), 200):
            for d in mpr.materials.summary.search(material_ids=keep[i0:i0+200],
                                                  fields=["material_id", "structure"]):
                raw[str(d.material_id)] = d.structure.as_dict()
    assert len(raw) > 200, f"only {len(raw)} competing phases -> the query looks wrong; do not build a hull on this"
    json.dump({"model": W.MODEL_TAG, "entries": raw}, open(CACHE, "w"))
    print("MP query returned", len(raw), "stable phases")
if MOCK_HULL:
    pool_struct = {}
else:
    _c = json.load(open(CACHE))
    # The hull currency must be the screening currency. The cache carries its model tag, so a leftover
    # file from another model can never be silently reused (invariant 6 on the hull side).
    assert _c.get("model") == W.MODEL_TAG, f"hull cache built with {_c.get('model')} != {W.MODEL_TAG} -> delete {CACHE}"
    pool_struct = _c["entries"]
    print("competing phases:", len(pool_struct))

# ---- re-relax them with the SAME model ----
# v1 died here: 1-atom elemental cells in the batch made the autobatcher try 255,066 atoms per batch.
# Expanding small cells to >=24 atoms and sorting by size keeps the batches homogeneous (v1 CELL 7-FIX).
items = []
for mid, sd in pool_struct.items():
    st = Structure.from_dict(sd)
    if len(st) < 24:
        n = math.ceil((24 / len(st)) ** (1/3)); st = st * (n, n, n)
    items.append(("HULL", mid, st))
items.sort(key=lambda x: len(x[2]))
t0 = time.time()
for i in range(0, len(items), 200):
    W.relax_batch(items[i:i+200], ckpt_main, done, save_cif=False)
    print(f"HULL {min(i+200, len(items))}/{len(items)} | {(time.time()-t0)/60:.1f} min", flush=True)

allE = W.merged_energies()
hdf = allE[allE.comp_id == "HULL"].set_index("tag")
entries = []
for mid, sd in pool_struct.items():
    if mid not in hdf.index: continue
    st = Structure.from_dict(sd)
    factor = float(hdf.loc[mid, "natoms"]) / len(st)      # the energy is for the EXPANDED cell
    entries.append(PDEntry(st.composition * factor, float(hdf.loc[mid, "energy"]), name=mid))
print("hull entries:", len(entries), "| skipped:", len(pool_struct) - len(entries))

# ---- Ehull, Ehull_eff, T* for every pristine O3 cell ----
R1 = W.recs_by_comp(allE[~allE.comp_id.isin(["REF", "HULL"])])
# Coverage guard: cell 7b should have blocked until PASS 1 finished; if cells were run out of
# order this catches the half-filled checkpoint instead of silently skipping compositions.
# (98%, not 100%: relax_batch drops individual structures that fail their own relaxation.)
_have = sum(1 for r in comps.itertuples() if {"O3_pris", "P3_pris"} <= set(R1.get(r.comp_id, {})))
assert _have >= 0.98 * len(comps), \
    f"PASS 1 incomplete: {_have}/{len(comps)} compositions have both pristine cells -> wait (cell 7b)"
print(f"pass-1 coverage: {_have}/{len(comps)}")

# 2026-09-04: the per-composition PhaseDiagram was the bottleneck (hours, and it killed the kernel).
# Every competitor is an MP *stable* phase, i.e. a hull vertex, so qhull's 8-D hull of ~200 vertices is
# the worst case for facet count and memory.  We never need the hull itself, only the hull energy at ONE
# composition, and that is a small linear programme:
#     E_hull(c) = min_x sum_i x_i e_i   s.t.  sum_i x_i f_i = f_c,  x_i >= 0
# (x_i = atom fraction taken from entry i, e_i = eV/atom, f = atomic-fraction vectors).  This is exactly
# what get_e_above_hull evaluates (checked to 1e-15 eV/atom on synthetic data; re-checked below on the
# first HULL_XCHECK real compositions).  Ehull = max(0, e_per_atom - E_hull), i.e. pymatgen semantics.
ENTRY_FRAC = [{str(el): en.composition.get_atomic_fraction(el) for el in en.composition.elements} for en in entries]
ENTRY_EPA  = np.array([en.energy_per_atom for en in entries])
_sub_cache = {}         # element set -> (entry indices, A matrix, element order); ~4,400 distinct sets over 7,073 rows

def hull_energy_per_atom(comp, key):
    if key not in _sub_cache:
        els = sorted(key)
        idx = [i for i, f in enumerate(ENTRY_FRAC) if set(f) <= key]
        A = np.array([[ENTRY_FRAC[i].get(el, 0.0) for i in idx] for el in els])
        _sub_cache[key] = (idx, A, els)
    idx, A, els = _sub_cache[key]
    b = np.array([comp.get_atomic_fraction(el) for el in els])
    res = linprog(ENTRY_EPA[idx], A_eq=A, b_eq=b, bounds=(0, None), method="highs")
    assert res.success, f"hull LP failed for {comp.reduced_formula}: {res.message}"
    return float(res.fun)

HULL_XCHECK = int(os.environ.get("HEO_HULL_XCHECK", 2))   # pymatgen cross-check on the first N compositions (0 = skip)
kT = W.K_B * W.T_SYNTH
rows = []
t0 = time.time()
for r in comps.itertuples():
    rec = R1.get(r.comp_id, {})
    if "O3_pris" not in rec or "P3_pris" not in rec: continue
    e = rec["O3_pris"]["energy"]; cnt = json.loads(r.site_counts)
    comp = Composition({**cnt, "Na": W.N_NA, "O": W.N_O})
    key = frozenset(str(el) for el in comp.elements)
    if MOCK_HULL:
        eh = W._mock_params(r.comp_id, cnt)["Ehull"]
    else:
        eh = max(0.0, e / comp.num_atoms - hull_energy_per_atom(comp, key))
        if len(rows) < HULL_XCHECK:
            _t = time.time()
            me = PDEntry(comp, e, name=r.comp_id)      # cell composition with the cell total energy;
            _sub = [entries[i] for i in _sub_cache[key][0]]     # pymatgen normalises per atom internally
            eh_pd = float(PhaseDiagram(_sub + [me]).get_e_above_hull(me))
            assert abs(eh_pd - eh) < 1e-6, f"LP hull {eh:.6f} != pymatgen {eh_pd:.6f} eV/atom for {r.comp_id}"
            print(f"  hull cross-check {r.comp_id}: LP {eh:.5f} == pymatgen {eh_pd:.5f} eV/atom "
                  f"({len(_sub)} competitors, PhaseDiagram took {time.time()-_t:.1f} s)", flush=True)
    s_atom = float(r.sconf_per_atom_kB)
    rows.append(dict(comp_id=r.comp_id, scheme=r.scheme, dopants=r.dopants, sconf_per_atom_kB=s_atom,
                     Ehull=eh, Ehull_eff=eh - kT*s_atom, T_star=eh/(W.K_B*s_atom),
                     dE_pris=(rec["P3_pris"]["energy"] - e)*1000/W.N_FU,
                     E_pris_O3=e, E_pris_P3=rec["P3_pris"]["energy"], Vol_pris_O3=rec["O3_pris"]["volume"]))
    if len(rows) % 500 == 0:
        HEARTBEAT.write_text(f"hull {len(rows)} {time.time():.0f}\n")     # keeps the idle watchdog off
        print(f"  Ehull {len(rows)} | {(time.time()-t0)/60:.1f} min", flush=True)
S = pd.DataFrame(rows)
tmp = f"{WORKDIR}/results/results_phase1.csv.tmp"
S.to_csv(tmp, index=False); os.replace(tmp, f"{WORKDIR}/results/results_phase1.csv")
print(f"Ehull for {len(S)} compositions in {(time.time()-t0)/60:.1f} min ({len(_sub_cache)} distinct element sets)")
print(S[["Ehull", "Ehull_eff", "T_star", "dE_pris"]].describe().round(3).to_string())
n_surv = int(((S.Ehull_eff < W.EHULL_EFF_CUT) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)).sum())
print(f"\nPASS 2 targets (Ehull_eff < {W.EHULL_EFF_CUT} AND pristine phase == O3): {n_surv} / {len(S)}")
for cut in (0.05, 0.10, 0.15):
    print(f"  Ehull_eff < {cut}: {int(((S.Ehull_eff < cut) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)).sum())}")


## 셀 9 — PASS 2 런처 (생존자 탈소듐, O3·P3에 같은 빈자리 패턴)

In [ ]:
# ============ CELL 9: PASS 2 LAUNCHER ============
# Survivors (Ehull_eff < cut AND pristine phase == O3) are read from results_phase1.csv by the worker.
launch(2)

## 셀 10 — 모니터 (PASS 2)

In [ ]:
# ============ CELL 10: MONITOR ============
monitor(2)

## 셀 10b — ★ 배리어: PASS 2 완료까지 대기 (Run All 안전장치)

In [ ]:
# ============ CELL 10b: BARRIER — block Run All until PASS 2 is complete ============
# Same as cell 7b: without this, "Run All" would let cell 11 judge phases from a partial PASS 2.
assert PROCS.get(2), "no PASS 2 workers tracked -> run the cell-9 launcher first"
monitor(2, wait=True)

## 셀 10c — PASS 3 타깃 + 런처 (v4: x = 22/27, 18/27 내부점)

`HEO_X4_SCOPE`로 범위를 정한다 (예산 로그 출력):
- `top500` (기본): base 실행(`HEO_BASE_RESULTS`)의 (tier, score) 상위 500 조성 — 팟 채점 v2 기준. score 열이 없으면 같은 정의로 재계산.
- `survivors`: 관문 통과 전체 (PASS 2와 같은 집합)
- `anchors`: 셀 5d만 (여기서는 no-op)
- `all`: 전 조성 (경고)

타깃은 어느 경우든 **이번 실행의 생존자와 교집합** — x\*는 x=0.52 값이 있어야 닫힌다.
워커는 `--pass-no 3`으로 `results/x4_targets.csv`를 읽어 `{ph}_x081_{k}`, `{ph}_x067_{k}` 20셀/조성을 이완한다
(같은 torch-sim 배치 경로·수렴 기준; 셀 5가 108/101/95원자 혼합 배치를 이미 실증했으므로 103/99원자 혼합도 그대로 통과).

In [ ]:
# ============ CELL 10c: PASS 3 TARGETS + LAUNCHER — interior Na points (v4, HEO_X4_SCOPE) ============
import os, json
import numpy as np, pandas as pd
X4_SCOPE = os.environ.get("HEO_X4_SCOPE", "top500")
S1 = pd.read_csv(f"{WORKDIR}/results/results_phase1.csv").set_index("comp_id")
surv = set(S1[(S1.Ehull_eff < W.EHULL_EFF_CUT) & (S1.dE_pris > W.DELTA_FLOOR_MEV_FU)].index)
comps_all = pd.read_csv(f"{WORKDIR}/results/compositions_master.csv")

def _score_like_pod(b):
    """The pod cell-11 ranking (2026-09-04 v2) recomputed from a results table without score/tier."""
    x0 = (W.N_NA - W.N_NA_REMOVE) / W.N_NA
    drop = b.dE_pris - b.dE_desod
    b = b.assign(x_cross=np.where(drop > 0, 1.0 - (1.0 - x0) * b.dE_pris / np.where(drop > 0, drop, 1.0), 0.0))
    b["score"] = b.x_cross.rank(ascending=True) + b.Ehull_eff.rank(ascending=True)
    b["tier"] = np.where(b.get("phase_desod", pd.Series(index=b.index, dtype=object)) == "O3", 0, 1)
    return b

if X4_SCOPE == "anchors":
    targets, how = [], "anchors only (cell 5d)"
elif X4_SCOPE == "survivors":
    targets, how = sorted(surv), "all gate survivors"
elif X4_SCOPE == "all":
    targets, how = list(comps_all.comp_id), "ALL compositions"
    print(f"[budget WARNING] scope=all: {len(targets)} compositions x 20 cells = {20*len(targets)} relaxations"
          " -- non-survivors have no x=0.52 point, so their x* cannot close; consider 'survivors'")
elif X4_SCOPE == "top500":
    BASE_RESULTS = os.environ.get("HEO_BASE_RESULTS", "")
    assert BASE_RESULTS and os.path.exists(BASE_RESULTS), \
        "HEO_X4_SCOPE=top500 needs HEO_BASE_RESULTS (a results_v3.csv with score/tier); otherwise use HEO_X4_SCOPE=survivors"
    b = pd.read_csv(BASE_RESULTS, index_col="comp_id").dropna(subset=["dE_desod"])
    if not {"score", "tier"} <= set(b.columns):
        print("  base results carry no score/tier -> recomputing the pod ranking (x_cross asc + Ehull_eff asc, tier 0 = O3 retained)")
        b = _score_like_pod(b)
    b = b.sort_values(["tier", "score"])
    targets, how = list(b.index[:500]), f"top-500 of {len(b)} by (tier, score) from {BASE_RESULTS}"
else:
    raise ValueError(f"HEO_X4_SCOPE={X4_SCOPE!r} not in top500|survivors|anchors|all")

n_before = len(targets)
targets = [t for t in targets if t in surv]
if X4_SCOPE != "all" and n_before != len(targets):
    print(f"  {n_before - len(targets)} target(s) are not gate survivors in THIS run -> dropped (x* needs the x=0.52 point)")
pd.DataFrame({"comp_id": targets, "scope": X4_SCOPE}).to_csv(f"{WORKDIR}/results/x4_targets.csv", index=False)
n_cells = len(targets) * 2 * 2 * W.N_VAC_SAMPLES
print(f"PASS 3 scope={X4_SCOPE}: {how} -> {len(targets)} compositions, {n_cells} relaxations "
      f"(~{n_cells/300/max(N_GPUS,1):.1f} h at the PASS-2 rate of ~300 comps/h/GPU)")
if targets:
    launch(3)
else:
    PROCS[3] = []; print("PASS 3: nothing to launch")

## 셀 10d — 모니터 (PASS 3)

In [ ]:
# ============ CELL 10d: MONITOR (PASS 3) ============
monitor(3) if PROCS.get(3) else print("PASS 3: no workers (scope anchors / empty targets)")

## 셀 10e — ★ 배리어: PASS 3 완료까지 대기

In [ ]:
# ============ CELL 10e: BARRIER — block Run All until PASS 3 is complete ============
if PROCS.get(3): monitor(3, wait=True)
else: print("PASS 3: nothing launched -> cell 11 will report the x4 columns as NaN")

## 셀 11 — Stage 3: 집계 · δ · 상 판정 · 채점 · 불변량 · audit

`dE_desod = min_k(E_P3,k − E_O3,k)` (짝지은 최소), `sigma_vac = std_k(E_P3,k − E_O3,k)`.
회색지대 `δ_i = max(σ_vac,i, |dE_A − dE_B|, 5)` — **대조 모델은 상 판정보다 반드시 앞에서** 읽습니다.
판정 뒤에 읽으면 δ를 갱신해도 `phase_desod`·`suppressed`·채점·저장 표·CIF가 전부 판정 이전 값으로 굳습니다.

불변량이 걸리면 실행은 멈추되 **진단용 표는 assert 앞에서 먼저 씁니다** — 계산은 못 믿어도 왜 걸렸는지는 봐야 합니다.
전부 통과하면 `_UNVALIDATED` 이름을 정식 이름으로 바꿉니다.

In [3]:
# ============ CELL 11: aggregate + judge + score + invariants + audit ============
import numpy as np, pandas as pd, json, hashlib, os, time
from pymatgen.core.periodic_table import Element

RULER = json.load(open(f"{WORKDIR}/results/ruler.json"))
assert RULER["model"] == W.MODEL_TAG, f"ruler built with {RULER['model']} != {W.MODEL_TAG}"
W.set_E_Na(RULER["E_Na"])

allE  = W.merged_energies()                                            # invariant 6 inside
comps = pd.read_csv(f"{WORKDIR}/results/compositions_master.csv").set_index("comp_id")
S     = pd.read_csv(f"{WORKDIR}/results/results_phase1.csv").set_index("comp_id")
COMP_HASH = hashlib.md5(open(f"{WORKDIR}/results/compositions_master.csv", "rb").read()).hexdigest()
survivors = S[(S.Ehull_eff < W.EHULL_EFF_CUT) & (S.dE_pris > W.DELTA_FLOOR_MEV_FU)].index

# ---- contrast model, loaded BEFORE the judgement ----
# Read after the judgement and delta would change nothing: phase_desod, suppressed, the score, the
# written tables and the exported CIFs would all stay frozen at their pre-contrast values.
CONTRAST = os.environ.get("HEO_CONTRAST_RESULTS", "")
DE_B = None
if CONTRAST and os.path.exists(CONTRAST):
    _B = pd.read_csv(CONTRAST, index_col="comp_id")
    assert "dE_desod" in _B.columns, f"{CONTRAST}: no dE_desod column -> not a results_v3.csv?"
    assert not _B.index.duplicated().any(), f"{CONTRAST}: duplicate comp_id"
    DE_B = _B["dE_desod"]
    print(f"contrast model: {int(DE_B.notna().sum())} compositions  <- {CONTRAST}")
else:
    print("no contrast results (set HEO_CONTRAST_RESULTS) - delta uses sigma_vac and the floor only")

need_tags = W.tags_for("all")
R2 = W.recs_by_comp(allE, survivors)
rows = [dict(W.metrics_from(rec), comp_id=cid)
        for cid, rec in ((c, R2.get(c, {})) for c in survivors)
        if all(t in rec for t in need_tags)]
M = pd.DataFrame(rows).set_index("comp_id")
print(f"complete compositions: {len(M)} / {len(survivors)} survivors")
assert len(M) >= 0.98 * max(len(survivors), 1), \
    f"PASS 2 incomplete: {len(M)}/{len(survivors)} survivors have a full tag set -> wait (cell 10b)"

floor = W.DELTA_FLOOR_MEV_FU
M["dE_desod_B"]     = DE_B.reindex(M.index) if DE_B is not None else np.nan
M["model_disagree"] = (M.dE_desod - M.dE_desod_B).abs()
M["delta"] = np.maximum.reduce([M.sigma_vac.values, M.model_disagree.fillna(0).values,
                                np.full(len(M), floor)])
M["phase_desod"] = [W.judge_phase(d, dl) for d, dl in zip(M.dE_desod, M.delta)]
M["dV_nat"]      = np.where(M.phase_desod == "O3", M.dV_O3, M.dV_trans)   # the volume of the phase that wins
M["suppressed"]  = (M.phase_desod == "O3")
print(M.phase_desod.value_counts().to_dict())
nf = int((M.n_phase_flip > 0).sum())
print(f"post-relax stacking: {len(M)-nf}/{len(M)} kept every label, {nf} have >=1 flipped structure")
if nf: print("  -> their dE_desod compares cells that are no longer O3 vs P3; treat with care.")
if DE_B is not None:
    print("model disagreement |dE_A - dE_B| (meV/f.u.):", M.model_disagree.describe().round(2).to_dict())

# 2026-09-04 (ranking): transition point by linear interpolation of dE(x) between x=1 and x=0.52.
# In the 0 K PBE currency essentially every composition is P3 at x=0.52 (first ORB run: 3,783 P3 /
# 70 AMB / 1 O3 of 3,854), exactly as both anchors -- including the experimentally delayed Zhao HEO --
# are.  "Retains O3 at 0.52" is therefore not a question this currency can answer; "who glides LAST"
# is.  x_cross is where dE crosses zero (smaller = more Na removed before gliding = better); it is the
# same quantity cell 5c reports for the calibration compounds, and it is what Zhao's "60 % of the
# capacity in O3" (x* ~ 0.71) measures.  ddE stays as a column but is no longer the score term:
# ddE = dE_desod - dE_pris punishes a large pristine O3 preference, which is the wrong direction.
X_DESOD = (W.N_NA - W.N_NA_REMOVE) / W.N_NA
def x_cross_of(dE_pris, dE_desod):
    drop = dE_pris - dE_desod
    return np.where(drop > 0, 1.0 - (1.0 - X_DESOD) * dE_pris / np.where(drop > 0, drop, 1.0), 0.0)
M["x_cross"] = x_cross_of(M.dE_pris.values, M.dE_desod.values)

# ---- capacity, ruler position, rank-sum score ----
def capacity_mAh_g(counts, n_Ni3, n_Mn3, n_Co2):
    """Redox electrons LEFT after the dopant accounting: Ni2+ -> Ni4+ is 2 e-, minus the sites already
    pushed to Ni3+ by low-valent dopants, plus Mn3+ -> Mn4+ and Co2+ -> Co3+.  V and Cu are not counted
    (undecided).  Capped at 1 e- per f.u. = one Na."""
    mass_fu = (Element("Na").atomic_mass + 2*Element("O").atomic_mass
               + sum(n*Element(e).atomic_mass for e, n in counts.items())/W.N_FU)
    e_avail = min(1.0, (2*counts.get("Ni", 0) - n_Ni3 + n_Mn3 + n_Co2)/W.N_FU)
    x_window = 1 - (W.N_NA - W.N_NA_REMOVE)/W.N_NA
    return 26801.0*e_avail/mass_fu, 26801.0*x_window/mass_fu

Rt = comps.join(S.drop(columns=[c for c in S.columns if c in comps.columns]), how="inner")
Rt = Rt.join(M.drop(columns=[c for c in M.columns if c in Rt.columns]), how="left")
Rt["cap_theor"], Rt["cap_window"] = zip(*[capacity_mAh_g(json.loads(r.site_counts), r.n_Ni3, r.n_Mn3, r.n_Co2)
                                          for r in Rt.itertuples()])
Rt["ruler_pos"] = ((Rt.dE_desod - RULER["dE_desod_HOST"])
                   / (RULER["dE_desod_HEO"] - RULER["dE_desod_HOST"]))      # 0 = HOST, 1 = HEO (Zhao 2020)
X_CROSS_HOST = float(x_cross_of(np.array([RULER["dE_pris_HOST"]]), np.array([RULER["dE_desod_HOST"]]))[0])
X_CROSS_HEO  = float(x_cross_of(np.array([RULER["dE_pris_HEO"]]),  np.array([RULER["dE_desod_HEO"]]))[0])
print(f"x_cross ruler: HOST {X_CROSS_HOST:.3f} | HEO {X_CROSS_HEO:.3f} | candidates median {Rt.x_cross.median():.3f}")

# 2026-09-04: signed volume changes of the two desodiated cells (diagnostic columns; the section-3
# quantities dV_O3 / dV_trans / dV_nat are unchanged).  Both cells shrink in-plane on TM oxidation; the
# prismatic gap of P3 (oxygen eclipsed across the gap) is wider than the octahedral gap of O3, so at the
# same x the P3 cell is the larger one and lands closer to the pristine volume.
_vol = allE.set_index(["comp_id", "tag"]).volume
Rt["Vol_desod_O3"] = [_vol.get((c, f"O3_desod_{int(k)}"), np.nan) if pd.notna(k) else np.nan
                      for c, k in zip(Rt.index, Rt.k_best)]
Rt["Vol_desod_P3"] = [_vol.get((c, f"P3_desod_{int(k)}"), np.nan) if pd.notna(k) else np.nan
                      for c, k in zip(Rt.index, Rt.k_best)]
Rt["sdV_O3"] = 100*(Rt.Vol_desod_O3 - Rt.Vol_pris_O3)/Rt.Vol_pris_O3
Rt["sdV_P3"] = 100*(Rt.Vol_desod_P3 - Rt.Vol_pris_O3)/Rt.Vol_pris_O3
_pair = Rt.dropna(subset=["Vol_desod_O3", "Vol_desod_P3"])
frac_P3_larger = float((_pair.Vol_desod_P3 > _pair.Vol_desod_O3).mean())
print(f"signed dV medians: O3 route {Rt.sdV_O3.median():+.2f}% | P3 route {Rt.sdV_P3.median():+.2f}% "
      f"| P3 cell larger than O3 cell at the same x: {frac_P3_larger:.2f} of {len(_pair)}")

# 2026-09-04 (ranking v2, user-approved): two tiers, one ordering function.
#   tier 0 = O3 retained at x=0.52 (suppressed) -- the screening's original question; always listed first.
#   tier 1 = everything else, ordered by rank-sum(x_cross asc, Ehull_eff asc).
# dV_nat is NO LONGER a score term: with every P3 composition it measures the accidental cancellation of
# c-expansion against a-contraction, and it penalised the one composition that actually kept O3 (real
# in-plane shrinkage 5.3 % vs ~1 % "cancelled" values).  It stays as a reported column.
SCORE_TERMS = {"x_cross": True, "Ehull_eff": True}   # True = ascending (lower is better)
def rank_sum(df):                        # ONE definition, used by the score and by the robustness check
    return sum(df[t].rank(ascending=asc) for t, asc in SCORE_TERMS.items())
def order(df):
    d = df.copy()
    d["score"] = rank_sum(d)
    d["tier"]  = np.where(d.suppressed == True, 0, 1)
    return d.sort_values(["tier", "score"])
cand = order(Rt.dropna(subset=["dE_desod"]))
top = cand.head(30)
n_tier0 = int((cand.tier == 0).sum())
print(f"candidates (judged): {len(cand)} | tier 0 (O3 retained at x=0.52): {n_tier0} -> always in top-30 "
      f"| tier-1 score range in top-30 {float(top[top.tier == 1].score.min()):.1f} - {float(top[top.tier == 1].score.max()):.1f}")

# ---- robustness ----
# Pessimistic shift: push dE_desod down by delta_mult*delta (the composition-specific noise), re-derive
# the phase, tier and x_cross, re-order.  Overlap with the base top-30 = how much of the list survives
# its own error bar.
def top_ids(df, delta_mult=1.0):
    d = df.dropna(subset=["dE_desod"]).copy()
    d["dE_desod"] = d.dE_desod - delta_mult*d.delta
    d["phase_desod"] = [W.judge_phase(x, dl) for x, dl in zip(d.dE_desod, d.delta)]
    d["suppressed"] = (d.phase_desod == "O3")
    d["x_cross"] = x_cross_of(d.dE_pris.values, d.dE_desod.values)
    d["dV_nat"] = np.where(d.phase_desod == "O3", d.dV_O3, d.dV_trans)
    return set(order(d).head(30).index)
base_top = set(top.index)
rob = {"n_candidates": len(cand), "n_tier0_suppressed_052": n_tier0, "n_AMB": int((Rt.phase_desod == "AMB").sum()),
       "delta_x1_overlap": len(base_top & top_ids(Rt, 1.0)) / max(1, len(base_top)),
       "delta_x2_overlap": len(base_top & top_ids(Rt, 2.0)) / max(1, len(base_top)),
       "inv4_frac_P3_cell_larger": frac_P3_larger,
       "x_cross_HOST": X_CROSS_HOST, "x_cross_HEO": X_CROSS_HEO,
       "top30_x_cross_range": [float(top.x_cross.min()), float(top.x_cross.max())]}
if DE_B is not None:
    rob["n_contrast"] = int(Rt.dE_desod_B.notna().sum())
    rob["model_disagree_median"] = float(Rt.model_disagree.median(skipna=True))
    _b = pd.read_csv(CONTRAST, index_col="comp_id")
    if "score" in _b.columns:
        rob["contrast_top30_overlap"] = len(base_top & set(_b.sort_values("score").head(30).index)) / max(1, len(base_top))
LEGACY = os.environ.get("HEO_LEGACY_TOP30", "")
if LEGACY and os.path.exists(LEGACY):
    leg = set(pd.read_csv(LEGACY).comp_id)
    rob["legacy_top30_overlap"] = len(base_top & leg) / max(1, len(leg))
print("robustness:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in rob.items()})

# ---- write the diagnostics FIRST, then assert ----
# A tripped invariant must stop the run, but it must not also destroy the evidence needed to fix it.
Rt = Rt.join(cand[["score", "tier"]], how="left")

# ======================= v4 block (CHANGE_SPEC_v4) — REPORTED COLUMNS ONLY; the score/tier above are untouched =======================
# (A) exact-cell decomposition of the x=0.52 volume change, (B) the two interior x points from PASS 3,
# the x* bracket, segment voltages and the x-direction formation energy.  Nothing here feeds the score.
X4_SCOPE = os.environ.get("HEO_X4_SCOPE", "top500")
E_NA = RULER["E_Na"]
_nan = float("nan")
DE_B_X = {}                                                   # contrast-model dE per interior x, if the contrast run was v4
if DE_B is not None:
    for s in W.X4_STEMS:
        if f"dE_{s}" in _B.columns: DE_B_X[s] = _B[f"dE_{s}"]
    print("contrast dE available at interior x:", sorted(DE_B_X) or "none -> delta(x) = max(sigma_vac(x), floor)")

X4M = {cid: {s: W.metrics_x(R2[cid], s) for s in W.X4_STEMS} for cid in M.index}
keys = []
for cid in M.index:
    kb = int(M.loc[cid, "k_best"]); keys += [(cid, "O3_pris"), (cid, f"O3_desod_{kb}"), (cid, f"P3_desod_{kb}")]
    for s in W.X4_STEMS:
        mx = X4M[cid][s]
        if mx: kb = mx[f"k_best_{s}"]; keys += [(cid, f"O3_{s}_{kb}"), (cid, f"P3_{s}_{kb}")]
_t = time.time()
CELLS = W.cells_bulk(keys, R2)
n_cell_missing = sum(v is None for v in CELLS.values())
print(f"cell matrices: {len(CELLS)} keys ({n_cell_missing} missing: no checkpoint cell and no CIF) in {time.time()-_t:.0f}s")

def _dv(cid, tag_d):
    cp, cd = CELLS.get((cid, "O3_pris")), CELLS.get((cid, tag_d))
    return W.dV_report(cp, cd) if (cp is not None and cd is not None) else None
DV_KEYS = ("dV_pct", "dA_pct", "dh_perp_pct", "d_inter_pris", "vm_strain")

v4rows = []
for cid in M.index:
    row = {"comp_id": cid}; kb = int(M.loc[cid, "k_best"]); ph = M.loc[cid, "phase_desod"]
    # ---- (A) the phase that wins at x=0.52, same branch as dV_nat; both phases kept for reference
    win = "O3" if ph == "O3" else "P3"
    r = _dv(cid, f"{win}_desod_{kb}")
    row.update(r if r else {k: _nan for k in DV_KEYS})
    for phx in ("O3", "P3"):
        rr = _dv(cid, f"{phx}_desod_{kb}")
        row[f"dV_pct_{phx}"] = rr["dV_pct"] if rr else _nan
        row[f"dh_perp_pct_{phx}"] = rr["dh_perp_pct"] if rr else _nan
        row[f"vm_strain_{phx}"] = rr["vm_strain"] if rr else _nan
    cp = CELLS.get((cid, "O3_pris"))
    if cp is not None:
        al, be, ga = W.cell_angles(cp); row["gamma_pris"] = ga; row["ab_dev_pris"] = max(abs(al - 90), abs(be - 90))
    else:
        row["gamma_pris"] = _nan; row["ab_dev_pris"] = _nan
    row["dV_legacy"] = float(M.loc[cid, "dV_nat"])       # v3 definition: |Vol_desod - Vol_pris| / Vol_pris from the checkpoint volume
    # ---- (B) verdict per x, descending: pris (gate 2 -> O3), x081, x067, desod (existing judgement)
    dE_p = float(M.loc[cid, "dE_pris"])
    verdicts = ["O3" if dE_p > floor else W.judge_phase(dE_p, floor)]
    E_nat = {"pris": float(M.loc[cid, "E_pris_O3"])}; src = {"pris": "O3"}
    for s in W.X4_STEMS:
        mx = X4M[cid][s]
        if not mx:
            row.update({f"dE_{s}": _nan, f"sigma_vac_{s}": _nan, f"delta_{s}": _nan, f"verdict_{s}": None,
                        f"k_best_{s}": _nan, f"dE_{s}_B": _nan})
            row.update({f"{k}_{s}": _nan for k in ("dV_pct", "dA_pct", "dh_perp_pct", "vm_strain")})
            verdicts.append(None); continue
        dB = float(DE_B_X[s].get(cid, _nan)) if s in DE_B_X else _nan
        delta_s = max(mx[f"sigma_vac_{s}"], 0.0 if np.isnan(dB) else abs(mx[f"dE_{s}"] - dB), floor)
        v = W.judge_phase(mx[f"dE_{s}"], delta_s)
        row.update({f"dE_{s}": mx[f"dE_{s}"], f"sigma_vac_{s}": mx[f"sigma_vac_{s}"], f"delta_{s}": delta_s,
                    f"verdict_{s}": v, f"k_best_{s}": mx[f"k_best_{s}"], f"dE_{s}_B": dB})
        verdicts.append(v)
        E_nat[s], src[s] = W.e_nat_of(v, mx[f"E_O3_{s}"], mx[f"E_P3_{s}"])
        rx = _dv(cid, f"{src[s][:2]}_{s}_{mx[f'k_best_{s}']}")
        row.update({f"{k}_{s}": (rx[k] if rx else _nan) for k in ("dV_pct", "dA_pct", "dh_perp_pct", "vm_strain")})
    verdicts.append(ph)
    E_nat["desod"], src["desod"] = W.e_nat_of(ph, float(M.loc[cid, "E_desod_O3"]), float(M.loc[cid, "E_desod_P3"]))
    row["verdict_pris"] = verdicts[0]
    row.update({f"E_nat_src_{s}": src.get(s) for s in W.X_GRID_DESC})
    row.update(W.x_star_bracket(verdicts))
    row.update(W.v_segments(E_nat, E_NA))
    row.update(W.formation_x(E_nat))
    # telescoping check (spec B-5): sum(dn_i V_seg_i)/13 must equal [E_nat(0.52) + 13 E_Na - E_pris]/13 exactly;
    # the difference to V_avg is the O3-path vs natural-path difference, reported, not a bug.
    if all(s in E_nat for s in W.X_GRID_DESC):
        vs = [row[f"V_seg_{W.X_LABEL[a]}_{W.X_LABEL[b]}"] for a, b in W.V_SEGMENTS]
        dn = [W.X_STEMS[b] - W.X_STEMS[a] for a, b in W.V_SEGMENTS]
        row["V_seg_avg"] = sum(d * v for d, v in zip(dn, vs)) / W.N_NA_REMOVE
        row["V_nat_avg"] = (E_nat["desod"] + W.N_NA_REMOVE * E_NA - E_nat["pris"]) / W.N_NA_REMOVE
    else:
        row["V_seg_avg"] = _nan; row["V_nat_avg"] = _nan
    v4rows.append(row)
V4 = pd.DataFrame(v4rows).set_index("comp_id")
Rt = Rt.join(V4.drop(columns=[c for c in V4.columns if c in Rt.columns]), how="left")

# ---- A-4 checks (reported; the invariant-9 assert sits with the other invariants below) ----
_ok = Rt.dropna(subset=["dV_pct"])
Rt["ident_resid_pp"] = 100 * ((1 + Rt.dV_pct/100) - (1 + Rt.dA_pct/100) * (1 + Rt.dh_perp_pct/100))
ident_bad = list(Rt.index[Rt.ident_resid_pp.abs() > 0.1])
_ang = _ok[((_ok.gamma_pris - 120).abs() <= 0.5) & (_ok.ab_dev_pris <= 0.5)]
_dvd = (_ang.dV_pct.abs() - _ang.dV_legacy).abs()
dv_legacy_bad = list(_dvd.index[_dvd > 0.05])
d_inter_out = list(_ok.index[(_ok.d_inter_pris < 4.8) | (_ok.d_inter_pris > 6.0)])
_vsr = (Rt.V_seg_avg - Rt.V_nat_avg).abs()
vseg_bad = list(_vsr.index[_vsr > 1e-3])
n_x4 = int(Rt.dE_x081.notna().sum()) if "dE_x081" in Rt else 0
xcls = Rt.x_star_class.value_counts(dropna=True).to_dict() if "x_star_class" in Rt else {}
print(f"\n[v4/A] dV decomposition on {len(_ok)}/{len(M)} judged compositions | identity residual > 0.1%p: {len(ident_bad)} "
      f"| angle-preserving rows {len(_ang)}, |dV_pct| vs dV_legacy > 0.05%p: {len(dv_legacy_bad)} | d_inter outside [4.8, 6.0] A: {len(d_inter_out)}")
if len(_ok):
    print("       medians (winning phase): dV %+.2f%%  dA %+.2f%%  dh_perp %+.2f%%  vm %.4f  d_inter %.3f A" % (
        _ok.dV_pct.median(), _ok.dA_pct.median(), _ok.dh_perp_pct.median(), _ok.vm_strain.median(), _ok.d_inter_pris.median()))
    print("       (a scalar dV that looks small is in-plane contraction cancelling interlayer expansion -- read dA and dh_perp)")
print(f"[v4/B] PASS 3 scope={X4_SCOPE}: interior points present for {n_x4} compositions | x_star_class: {xcls}")
if vseg_bad: print(f"  [WARNING] V_seg telescoping identity broken (> 1 mV) for {len(vseg_bad)}: {vseg_bad[:5]}")
if "V_seg_avg" in Rt:
    _pd = (Rt.V_seg_avg - Rt.V_avg).dropna()
    if len(_pd): print(f"       V_seg-weighted average vs V_avg (O3 path): median {_pd.median()*1000:+.1f} mV -- non-zero where P3 wins, by definition")
if ident_bad:   print("  [WARNING] identity residual candidates:", ident_bad[:10])
if d_inter_out: print("  [WARNING] d_inter_pris outside the layered window:", d_inter_out[:10])

# top-30 overlap by the two dV definitions (both from the same exact volume -> identical by construction; recorded
# for the spec's A-4 list) plus the informative one: |dV| ordering vs von-Mises ordering among the judged set.
_j = Rt.dropna(subset=["dV_legacy", "dV_pct"])
_t_leg = set(_j.dV_legacy.nsmallest(30).index); _t_new = set(_j.dV_pct.abs().nsmallest(30).index)
_t_vm  = set(_j.dropna(subset=["vm_strain"]).vm_strain.nsmallest(30).index)
rob["top30_dV_legacy_vs_new_overlap"] = len(_t_leg & _t_new) / max(1, len(_t_leg))
rob["top30_absdV_vs_vm_strain_overlap"] = len(_t_leg & _t_vm) / max(1, len(_t_leg))
V4_AUDIT = dict(x4_scope=X4_SCOPE, n_x4_targets=int(pd.read_csv(f"{WORKDIR}/results/x4_targets.csv").shape[0])
                if os.path.exists(f"{WORKDIR}/results/x4_targets.csv") else 0,
                n_x4_complete=int(((Rt.dE_x081.notna()) & (Rt.dE_x067.notna())).sum()) if "dE_x081" in Rt else 0,
                n_cell_missing=n_cell_missing, x_star_class_counts=xcls,
                n_ident_bad=len(ident_bad), n_dv_legacy_mismatch=len(dv_legacy_bad), n_angle_preserving=len(_ang),
                n_d_inter_out=len(d_inter_out), n_vseg_identity_bad=len(vseg_bad),
                convexity_counts=Rt.convexity_flag.value_counts(dropna=True).to_dict() if "convexity_flag" in Rt else {},
                dV_note="checkpoint volume was already |det L| (ASE get_volume); dV_legacy == |dV_pct| by construction, "
                        "the new content is the dA / dh_perp / von Mises decomposition",
                base_workdir=W.BASE_WORKDIR, pkg_versions=(json.load(open(f"{WORKDIR}/results/pkg_versions.json"))
                                                          if os.path.exists(f"{WORKDIR}/results/pkg_versions.json") else {}),
                x_grid={s: round(W.X_OF[s], 4) for s in W.X_GRID_DESC}, vac_salts=W.VAC_SALT,
                score_note="score/tier definitions identical to the 2026-09-04 v2 cell; every v4 quantity is a reported column")
top = Rt.loc[top.index]            # the saved top-30 carries the v4 columns too (order/score unchanged)
# ======================= end of the v4 block =======================
Rt.to_csv(f"{WORKDIR}/results/results_v3_UNVALIDATED.csv")
top.to_csv(f"{WORKDIR}/results/top30_v3_UNVALIDATED.csv")
audit = dict(model=W.MODEL_TAG, compositions_md5=COMP_HASH, n_compositions=len(comps), n_pass1=len(S),
             n_survivors=len(survivors), n_complete=len(M), n_candidates=len(cand), n_tier0_suppressed_052=n_tier0,
             phase_desod_counts=Rt.phase_desod.value_counts(dropna=True).to_dict(),
             ruler=RULER, robustness=rob, ehull_eff_cut=W.EHULL_EFF_CUT, delta_floor=floor,
             sqs_cutoffs=W.SQS_CUTOFFS, sqs_steps=W.SQS_MC_STEPS, n_vac_samples=W.N_VAC_SAMPLES,
             dE_desod_definition="paired minimum  min_k(E_P3,k - E_O3,k)",
             sigma_vac_definition="std_k(E_P3,k - E_O3,k)",
             x_cross_definition="1 - (1 - 14/27) * dE_pris / (dE_pris - dE_desod)  (linear, 2-point)",
             score_definition="tier 0 = O3 retained at x=0.52 (always first); tier 1 ordered by rank-sum of "
                              "x_cross asc, Ehull_eff asc; dV_nat reported only (revised 2026-09-04 v2)",
             invariant4_definition="Vol(P3_desod,k_best) > Vol(O3_desod,k_best) for the majority (revised 2026-09-04)",
             lattice=dict(a=W.A_HEX, c=W.C_HEX, z_O=W.Z_O), E_Na=RULER["E_Na"],
             run_label=os.environ.get("HEO_RUN_LABEL", W.MODEL_TAG),
             orb_ckpt=os.environ.get("HEO_ORB_CKPT", ""),
             worker_md5=hashlib.md5(open(W.__file__, "rb").read()).hexdigest(),
             timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
audit["v4"] = V4_AUDIT
json.dump(audit, open(f"{WORKDIR}/results/audit.json", "w"), indent=1, default=str)

V = Rt.dropna(subset=["V_avg"])
assert (V.V_avg > 0).all(), "invariant 1: negative voltage -> sign bug"
assert V.dE_desod.abs().max() < 500, "invariant 2: |dE| > 500 meV/f.u. -> per-cell instead of per-f.u.?"
# invariant 9 (v4): every segment voltage on the 4-point grid is positive (invariant 1 per segment)
_vseg = Rt[[c for c in Rt.columns if c.startswith("V_seg_") and c.count("_") == 3]].dropna(how="all")
assert (_vseg.fillna(1.0) > 0).all().all(), f"invariant 9: non-positive segment voltage\n{_vseg[(_vseg <= 0).any(axis=1)].head()}"
_x4v = Rt[["dE_x081", "dE_x067"]].dropna()
assert _x4v.abs().max().max() < 500 if len(_x4v) else True, "invariant 2 at interior x: |dE| > 500 meV/f.u."
# invariant 4 (revised 2026-09-04, user-approved).  The original "dV_trans >= dV_O3 for most P3
# compositions" assumed the transition adds net volume strain; the anchors (HOST 4.42 vs 1.84 %, HEO
# 2.61 vs 1.24 %) and the full ORB run show the opposite, for the reason given above.  What the
# invariant guards -- swapped O3/P3 labels or a volume-bookkeeping error -- is tested directly:
# with the same vacancy pattern the prismatic cell must be the larger one.
assert frac_P3_larger > 0.5, \
    f"invariant 4: the P3 cell should be larger than the O3 cell at the same x for most compositions (got {frac_P3_larger:.2f})"
assert COMP_HASH == hashlib.md5(open(f"{WORKDIR}/results/compositions_master.csv", "rb").read()).hexdigest(), \
    "invariant 7: compositions_master.csv changed during the run"
print("invariants 1,2,4,7,9 OK   (3 in cell 4, 5 in cell 5, 6+10 in every merged_energies call, 8 in build_items)")

for a, b in (("results_v3_UNVALIDATED.csv", "results_v3.csv"), ("top30_v3_UNVALIDATED.csv", "top30_v3.csv")):
    os.replace(f"{WORKDIR}/results/{a}", f"{WORKDIR}/results/{b}")
print("written:", sorted(os.listdir(f"{WORKDIR}/results")))
print(top[["tier", "scheme", "V_avg", "dE_pris", "dE_desod", "x_cross", "phase_desod", "Ehull_eff", "T_star",
           "dV_nat", "cap_theor"]].round(3).head(10).to_string())
_v4c = [c for c in ["x_star_class", "x_star_mid", "verdict_x081", "verdict_x067", "dV_pct", "dA_pct", "dh_perp_pct",
                    "vm_strain", "V_seg_100_081", "V_seg_067_052", "Ef_x081", "Ef_x067", "convexity_flag"] if c in Rt.columns]
print("\nv4 columns of the same top-10:")
print(Rt.loc[top.index[:10], _v4c].round(3).to_string())


ModuleNotFoundError: No module named 'pandas'

## 셀 12 — top-N 내보내기 (DFT Tier-1 입력)

`HEO_SAVE_RELAXED=1`이라 이완 구조 CIF는 계산할 때 **이미 저장돼 있습니다.** 여기서는 다시 이완하지 않고 색인만 만듭니다 —
재이완하면 캐시 에너지와 미세하게 달라져 저장된 표와 내보낸 구조가 어긋납니다.
`k_best`는 `dE_desod`를 결정한 바로 그 빈자리 샘플이므로, O3와 P3가 **같은 Na 배열**을 쓴 짝을 내보냅니다.

In [ ]:
# ============ CELL 12: export the top-N structures (DFT Tier-1 input) ============
# HEO_SAVE_RELAXED=1 means every relaxed cell was already written to CIF while it was computed, so
# nothing is re-relaxed here.  Re-relaxing would produce slightly different energies than the ones
# already in results_v3.csv, and the exported structures would no longer match the reported numbers.
import shutil, json, os
import numpy as np, pandas as pd
top = pd.read_csv(f"{WORKDIR}/results/top30_v3.csv", index_col=0)
OUT = f"{WORKDIR}/results/top30_cifs"; os.makedirs(OUT, exist_ok=True)
index, missing = {}, []
for cid in top.index:
    kb = int(top.loc[cid, "k_best"])          # the vacancy sample that set dE_desod: O3 and P3 share it
    files = {"O3_pris": f"{cid}__O3_pris.cif", "P3_pris": f"{cid}__P3_pris.cif",
             "O3_desod": f"{cid}__O3_desod_{kb}.cif", "P3_desod": f"{cid}__P3_desod_{kb}.cif"}
    got = {}
    for role, fn in files.items():
        src = W.relaxed_cif_path(cid, fn[len(cid) + 2:-4])        # own workdir first, then the read-only base (v4)
        if src: shutil.copy(src, f"{OUT}/{fn}"); got[role] = fn
        else: missing.append(f"{cid}/{role}")
    index[cid] = dict(k_best=kb, files=got, **{c: (float(top.loc[cid, c]) if c in top.columns else None)
                      for c in ("dE_desod", "ddE", "Ehull_eff", "T_star", "V_avg", "dV_nat",
                                "ruler_pos", "cap_theor", "delta", "sigma_vac")})
json.dump(index, open(f"{WORKDIR}/results/top30_cif_index.json", "w"), indent=1)
print(f"exported {len(index)} compositions x 4 cells -> {OUT}")
if missing:
    print("MISSING:", missing[:10], "..." if len(missing) > 10 else "")
    print("  (HEO_SAVE_RELAXED was off for part of the run; re-run the launcher for those compositions)")

# ORB confidence head: the batch engine cannot read it, so take a single point for the top-N only.
if MODE == "orb":
    from pymatgen.core import Structure as _St
    from pymatgen.io.ase import AseAtomsAdaptor
    from orb_models.forcefield import pretrained
    try:
        from orb_models.forcefield.inference.calculator import ORBCalculator   # orb-models >= 0.6
    except ImportError:
        from orb_models.forcefield.calculator import ORBCalculator             # <= 0.5 layout
    _l = getattr(pretrained, W.ORB_MODEL)(device=W.DEVICE)
    _raw, _ad = _l if isinstance(_l, tuple) else (_l, None)
    try:
        _calc = ORBCalculator(_raw, device=W.DEVICE) if _ad is None else \
                ORBCalculator(_raw, atoms_adapter=_ad, device=W.DEVICE)
    except TypeError:
        _calc = ORBCalculator(_raw, device=W.DEVICE)
    conf = {}
    for cid, meta in index.items():
        fn = meta["files"].get("O3_desod")
        if not fn: continue
        at = AseAtomsAdaptor.get_atoms(_St.from_file(f"{OUT}/{fn}")); at.calc = _calc
        at.get_potential_energy(); c = at.calc.results.get("confidence")
        conf[cid] = (float(np.mean(c)), float(np.min(c))) if c is not None else (float("nan"),)*2
    if conf:
        top["conf_mean"] = [conf.get(c, (np.nan, np.nan))[0] for c in top.index]
        top["conf_min"]  = [conf.get(c, (np.nan, np.nan))[1] for c in top.index]
        top.to_csv(f"{WORKDIR}/results/top30_v3.csv")
        print("confidence head read for", len(conf), "compositions |",
              "min conf_min =", round(float(np.nanmin(top.conf_min)), 3))

## 셀 12b — (선택) base ↔ 현재 실행 대조 (파인튜닝 A/B)

파인튜닝한 ORB로 재실행할 때 **별도 `HEO_WORKDIR`** 을 쓰고, `HEO_ORB_CKPT=<가중치 파일>`과
`HEO_BASE_RESULTS=<base 실행의 results_v3.csv>` 를 export 하면 이 셀이 두 실행을 조성 단위로 짝지어 비교합니다.
구조 생성이 내용 주소(comp_rng) 결정론이라 **두 실행은 완전히 같은 SQS 셀·같은 빈자리 패턴**을 이완합니다 —
아래 차이는 전부 모델 차이입니다.

찍는 것: 부호 일치율(§8 "부호 일치 측정"), base 대비 기울기(§8 "연화 계수" — 1보다 크면 파인튜닝이 에너지 차이를 키움),
Spearman 순위 상관(랭킹 보존), phase 판정 혼동 행렬, suppressed·top-30(ddE 기준) 겹침, 앵커 gap 변화.
결과는 `results/compare_base.{csv,json}`에 저장되어 셀 13 아카이브에 포함됩니다. base 실행에서는 그냥 스킵됩니다.

추가 기록(2026-09-03): 부호 있는 평균 이동(`mean_shift`), phase 뒤집힘 원인 분해(부호 vs δ), scheme별 일치율,
실제 `top30_v3.csv`(rank-sum 점수) 겹침, 앵커 HOST/HEO `dE_desod`·`E_Na` 상세.


In [ ]:
# ============ CELL 12b: base-vs-current run comparison (fine-tuning A/B) ============
# Optional.  HEO_BASE_RESULTS points at a previous run's results_v3.csv (e.g. stock ORB); the
# current run must live in its OWN HEO_WORKDIR with a distinct MODEL_TAG (HEO_ORB_CKPT sets one).
# comp_rng() content-addresses SQS and vacancy patterns, so both runs relaxed identical inputs:
# every difference measured here is the model, not the sampling.
import os, json
import numpy as np, pandas as pd

BASE = os.environ.get("HEO_BASE_RESULTS", "")
if not (BASE and os.path.exists(BASE)):
    print("no base run to compare (export HEO_BASE_RESULTS=<base results_v3.csv>) - skipping")
else:
    cur  = pd.read_csv(f"{WORKDIR}/results/results_v3.csv", index_col="comp_id")
    base = pd.read_csv(BASE, index_col="comp_id")
    both = cur.join(base, how="inner", rsuffix="_base")
    print(f"current {len(cur)}  x  base {len(base)}  ->  common: {len(both)} compositions")
    assert len(both), "no common compositions -> wrong HEO_BASE_RESULTS?"

    stats = {"n_current": len(cur), "n_base": len(base), "n_common": len(both),
             "model_current": W.MODEL_TAG, "base_results": BASE}
    for q in ("dE_pris", "dE_desod", "ddE", "V_avg", "Ehull_eff"):
        if q not in both or f"{q}_base" not in both: continue
        a, b = both[q], both[f"{q}_base"]
        ok = a.notna() & b.notna()
        if not ok.any(): continue
        stats[q] = dict(
            slope_vs_base=round(float((a[ok]*b[ok]).sum() / max((b[ok]**2).sum(), 1e-12)), 3),
            pearson=round(float(a[ok].corr(b[ok])), 3),
            spearman=round(float(a[ok].corr(b[ok], method="spearman")), 3),
            sign_agree=round(float((np.sign(a[ok]) == np.sign(b[ok])).mean()), 3),
            mean_shift=round(float((a[ok] - b[ok]).mean()), 2),      # signed: which WAY the model moved
            mean_abs_shift=round(float((a[ok] - b[ok]).abs().mean()), 2))
    # slope > 1: the current model AMPLIFIES energy differences vs base (de-softening);
    # spearman ~ 1 with slope != 1: ranking preserved, only the scale moved (Deng 2025 picture).

    # per-scheme sign agreement: uneven fine-tuning coverage shows up here before it shows up
    # in the global mean (scheme C is small and chemically distinct - watch it).
    if "scheme" in both and "dE_desod_base" in both:
        per = {}
        for s, g in both.groupby("scheme"):
            a, b = g.dE_desod, g.dE_desod_base
            ok = a.notna() & b.notna()
            if ok.sum() < 5: continue
            per[str(s)] = dict(n=int(ok.sum()),
                               sign_agree=round(float((np.sign(a[ok]) == np.sign(b[ok])).mean()), 3),
                               slope_vs_base=round(float((a[ok]*b[ok]).sum() / max((b[ok]**2).sum(), 1e-12)), 3))
        stats["dE_desod_by_scheme"] = per

    conf = pd.crosstab(both.phase_desod, both.phase_desod_base, dropna=False)
    print("\nphase_desod confusion (rows=current, cols=base):\n", conf.to_string())
    stats["phase_confusion"] = {f"{r}|{c}": int(conf.loc[r, c]) for r in conf.index for c in conf.columns}

    # why did a judgement flip: the dE SIGN changed (the models disagree on the physics), or the
    # sign held and only delta moved the AMB boundary (uncertainty bookkeeping, not physics)?
    okp  = both.phase_desod.notna() & both.phase_desod_base.notna()
    flip = okp & (both.phase_desod != both.phase_desod_base)
    sgn  = np.sign(both.dE_desod) != np.sign(both.dE_desod_base)
    stats["phase_flips"] = dict(total=int(flip.sum()),
                                sign_driven=int((flip & sgn).sum()),
                                delta_driven=int((flip & ~sgn).sum()))

    sup_c = set(both.index[both.suppressed == True]); sup_b = set(both.index[both.suppressed_base == True])
    stats["suppressed"] = dict(current=len(sup_c), base=len(sup_b),
                               gained=len(sup_c - sup_b), lost=len(sup_b - sup_c),
                               jaccard=round(len(sup_c & sup_b) / max(len(sup_c | sup_b), 1), 3))
    topN = 30
    t_c = set(both.ddE.nlargest(topN).index); t_b = set(both.ddE_base.nlargest(topN).index)
    stats[f"top{topN}_ddE_overlap"] = len(t_c & t_b)

    # the DFT-bound list is the rank-sum top-30 (cell 11), not the ddE top-30: compare the
    # actually saved files so this number answers "how many Tier-1 candidates change?"
    bt = os.path.join(os.path.dirname(BASE), "top30_v3.csv")
    ct = f"{WORKDIR}/results/top30_v3.csv"
    if os.path.exists(bt) and os.path.exists(ct):
        s_b = set(pd.read_csv(bt).comp_id); s_c = set(pd.read_csv(ct).comp_id)
        stats["top30_score_overlap"] = len(s_b & s_c)

    br = os.path.join(os.path.dirname(BASE), "ruler.json")
    if os.path.exists(br):
        rb = json.load(open(br)); rc = json.load(open(f"{WORKDIR}/results/ruler.json"))
        g = lambda d, k: round(float(d.get(k, float("nan"))), 2)
        stats["anchor_gap"] = dict(base=g(rb, "gap"), current=g(rc, "gap"),
                                   dE_desod_HOST=[g(rb, "dE_desod_HOST"), g(rc, "dE_desod_HOST")],
                                   dE_desod_HEO=[g(rb, "dE_desod_HEO"), g(rc, "dE_desod_HEO")],
                                   E_Na=[round(float(rb.get("E_Na", float("nan"))), 4),
                                         round(float(rc.get("E_Na", float("nan"))), 4)],
                                   base_model=rb.get("model"), current_model=rc.get("model"))

    keep = [c for c in ("scheme", "dE_pris", "dE_pris_base", "dE_desod", "dE_desod_base",
                        "ddE", "ddE_base", "delta", "delta_base", "phase_desod", "phase_desod_base",
                        "suppressed", "suppressed_base", "V_avg", "V_avg_base",
                        "Ehull_eff", "Ehull_eff_base", "ruler_pos", "ruler_pos_base") if c in both]
    both[keep].to_csv(f"{WORKDIR}/results/compare_base.csv")
    json.dump(stats, open(f"{WORKDIR}/results/compare_base.json", "w"), indent=1, default=str)
    print("\ncomparison:", json.dumps({k: v for k, v in stats.items()
                                        if k not in ("phase_confusion", "base_results")}, indent=1, default=str))
    print("written: results/compare_base.csv, results/compare_base.json")


## 셀 12c — v4 요약 리포트 (`results/report_v4.md`)

스펙 §5-3의 자동 리포트: A 검산 표, x_star_class 분포, V_seg 검산, 앵커 파일럿, top-30의 v4 열.

In [ ]:
# ============ CELL 12c: v4 summary report -> results/report_v4.md ============
import os, json
import numpy as np, pandas as pd
Rv = pd.read_csv(f"{WORKDIR}/results/results_v3.csv", index_col="comp_id")
aud = json.load(open(f"{WORKDIR}/results/audit.json")); v4 = aud.get("v4", {})
top = pd.read_csv(f"{WORKDIR}/results/top30_v3.csv", index_col=0)
L = []
L.append(f"# report_v4 — {aud['model']} | {aud['timestamp']} | scope {v4.get('x4_scope')}\n")
L.append("## 1. 변경 A — 부피 분해 검산 (스펙 A-4)\n")
L.append("| 항목 | 값 |\n|---|---|")
_j = Rv.dropna(subset=["dV_pct"])
L.append(f"| 분해 계산된 조성 | {len(_j)} / {int(Rv.dE_desod.notna().sum())} (cell 누락 {v4.get('n_cell_missing')}) |")
L.append(f"| 항등식 잔차 > 0.1 %p | {v4.get('n_ident_bad')} (기준 < 1 % → {'OK' if v4.get('n_ident_bad', 0) < 0.01*max(len(_j),1) else 'FAIL'}) |")
L.append(f"| 각도 보존 행(γ 120±0.5°, α,β 90±0.5°) | {v4.get('n_angle_preserving')} 중 dV_new≠dV_legacy(>0.05 %p) {v4.get('n_dv_legacy_mismatch')} |")
L.append(f"| d_inter_pris ∉ [4.8, 6.0] Å | {v4.get('n_d_inter_out')} |")
L.append(f"| top-30 겹침 legacy dV vs new dV | {aud['robustness'].get('top30_dV_legacy_vs_new_overlap')} (정의상 1.0 — 아래 주) |")
L.append(f"| top-30 겹침 \\|dV\\| vs von Mises | {aud['robustness'].get('top30_absdV_vs_vm_strain_overlap')} |")
if len(_j):
    q = _j[["dV_pct", "dA_pct", "dh_perp_pct", "vm_strain", "d_inter_pris"]].describe().loc[["25%", "50%", "75%"]].round(3)
    L.append("\n이긴 상 기준 분위수:\n\n" + q.to_markdown() + "\n")
    for phx in ("O3", "P3"):
        c = _j[[f"dV_pct_{phx}", f"dh_perp_pct_{phx}", f"vm_strain_{phx}"]].median().round(3).to_dict()
        L.append(f"- {phx} 경로 중앙값: {c}")
L.append(f"\n> 주: {v4.get('dV_note')}\n")
L.append("## 2. 변경 B — x\\* 분포\n")
L.append("| x_star_class | n |\n|---|---|")
for k, n in (v4.get("x_star_class_counts") or {}).items(): L.append(f"| {k} | {n} |")
L.append(f"\n내부점 계산 완료 조성: {v4.get('n_x4_complete')} / 타깃 {v4.get('n_x4_targets')}\n")
if "x_star_mid" in Rv and Rv.x_star_mid.notna().any():
    L.append(f"x_star_mid 중앙값 {Rv.x_star_mid.median():.3f} (transition/amb_boundary 만) | convexity {v4.get('convexity_counts')}\n")
    hv = Rv.dropna(subset=["dE_x081", "dE_x067"])
    L.append("verdict 조합 (x100/x081/x067/x052) 상위:\n")
    combo = (hv.verdict_pris.astype(str) + "/" + hv.verdict_x081.astype(str) + "/" + hv.verdict_x067.astype(str) + "/" + hv.phase_desod.astype(str)).value_counts().head(8)
    L.append("| 조합 | n |\n|---|---|"); L += [f"| {k} | {n} |" for k, n in combo.items()]
L.append("\n## 3. V_seg 검산\n")
if "V_seg_avg" in Rv:
    r = (Rv.V_seg_avg - Rv.V_nat_avg).abs().dropna()
    p = (Rv.V_seg_avg - Rv.V_avg).dropna() * 1000
    L.append(f"- 텔레스코핑 항등식 |Σ dn·V_seg/13 − V_nat_avg| : max {r.max() if len(r) else float('nan'):.2e} V, > 1 mV 인 조성 {v4.get('n_vseg_identity_bad')}")
    if len(p): L.append(f"- V_seg 가중평균 − V_avg(O3 경로): 중앙값 {p.median():+.1f} mV (P3가 이기는 조성에서 정의상 음수)")
    vs = Rv[[c for c in Rv.columns if c.startswith("V_seg_") and c.count("_") == 3]].dropna()
    if len(vs): L.append("- 구간 전압 중앙값 (V): " + ", ".join(f"{c} {vs[c].median():.3f}" for c in vs.columns) + f" | 불변량 9 최소값 {vs.min().min():.3f}")
L.append("\n## 4. 앵커 x4 파일럿\n")
fa = f"{WORKDIR}/results/anchors_x4.csv"
if os.path.exists(fa):
    ax = pd.read_csv(fa, index_col=0)
    cols = [c for c in ["dE_pris", "dE_x081", "dE_x067", "dE_desod", "verdict_x081", "verdict_x067", "verdict_desod",
                        "x_star_class", "x_star_lo", "x_star_hi", "V_seg_100_081", "V_seg_081_067", "V_seg_067_052", "Ef_x081", "Ef_x067"] if c in ax]
    L.append(ax[cols].round(3).T.to_markdown() + "\n")
    L.append("HEO−HOST 순서는 진단(0 K 화폐에 지연 신호 없음: heo_v3.md §3).")
L.append("\n## 5. top-30 (채점 v2 불변) 의 v4 열\n")
tc = [c for c in ["tier", "x_cross", "x_star_class", "x_star_mid", "dV_pct", "dA_pct", "dh_perp_pct", "vm_strain", "Ef_x081", "Ef_x067", "convexity_flag"] if c in top]
L.append(top[tc].round(3).to_markdown() + "\n")
L.append(f"\n패키지: {v4.get('pkg_versions')}  |  base: {v4.get('base_workdir')}\n")
out = f"{WORKDIR}/results/report_v4.md"
open(out, "w").write("\n".join(L))
print("written:", out); print("\n".join(L[:14]))

## 셀 13 — 아카이브하고 팟 종료 (마지막)

순서가 중요합니다: 결과 → 아카이브 → (업로드) → `DONE` → 종료.
`/workspace`가 Network Volume이므로 `delete`도 안전합니다(데이터는 볼륨에 남습니다). 그래도 기본은 `stop`으로 두었으니 삭제까지 원하시면 `HEO_TERMINATE_MODE=delete`를 export하세요.

In [ ]:
# ============ CELL 13: archive + terminate ============
# Order matters: results -> archive -> (upload) -> DONE sentinel -> terminate.
# Every external binary is optional; a missing one must never skip the REST fallback, or the pod bills on.
import tarfile, subprocess, shutil, urllib.request, json, time, os

def _run(cmd, **kw):
    try: return subprocess.run(cmd, **kw)
    except (FileNotFoundError, PermissionError, OSError) as e:
        print(f"  {cmd[0]} unavailable: {e!r}"); return None

def pod_terminate(mode):
    pod = os.environ.get("RUNPOD_POD_ID")
    if not pod: return False, "RUNPOD_POD_ID not set (nothing to terminate)"
    cmds = ([["runpodctl", "pod", "delete", pod], ["runpodctl", "remove", "pod", pod]] if mode == "delete"
            else [["runpodctl", "pod", "stop", pod], ["runpodctl", "stop", "pod", pod]])
    for c in cmds:
        r = _run(c, capture_output=True)
        if r is not None and r.returncode == 0: return True, " ".join(c)
    key = os.environ.get("RUNPOD_API_KEY")
    if not key: return False, "runpodctl failed and RUNPOD_API_KEY not set"
    url = f"https://rest.runpod.io/v1/pods/{pod}" + ("" if mode == "delete" else "/stop")
    req = urllib.request.Request(url, method="DELETE" if mode == "delete" else "POST",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        urllib.request.urlopen(req, timeout=20); return True, "REST " + url
    except Exception as e:
        return False, f"REST failed: {e!r}"

mode = os.environ.get("HEO_TERMINATE_MODE", "stop")
tgz = f"{WORKDIR}/heo_v3_results_{time.strftime('%Y%m%d_%H%M')}.tgz"
with tarfile.open(tgz, "w:gz") as tf:
    for sub in ("results", "checkpoints", "hull_cache"):
        if os.path.exists(f"{WORKDIR}/{sub}"): tf.add(f"{WORKDIR}/{sub}", arcname=sub)
print("archive:", tgz, f"{os.path.getsize(tgz)/1e6:.1f} MB")
print("  (structures/ is left out: it is large and stays on the Network Volume)")

uploaded = False
remote = os.environ.get("HEO_RCLONE_REMOTE")
if remote and shutil.which("rclone"):
    r = _run(["rclone", "copy", tgz, remote]); uploaded = (r is not None and r.returncode == 0)
    print("uploaded:", uploaded)

hook = os.environ.get("HEO_WEBHOOK")
if hook:
    try:
        urllib.request.urlopen(urllib.request.Request(
            hook, data=json.dumps({"text": f"[HEO v3] run finished, terminate={mode}, uploaded={uploaded}"}).encode(),
            headers={"Content-Type": "application/json"}), timeout=10)
    except Exception as e:
        print("webhook failed:", e)

open(f"{WORKDIR}/results/DONE", "w").write(json.dumps({"mode": mode, "uploaded": uploaded, "t": time.time()}))
print("terminate mode:", mode, "| pod:", os.environ.get("RUNPOD_POD_ID"))
if mode in ("delete", "stop"):
    time.sleep(5)                                   # let file writes flush
    ok, how = pod_terminate(mode)
    print(f"pod {mode}: {'issued via ' + how if ok else 'FAILED -> ' + how}")
    if not ok:
        print("  !! the cell-1b watchdog retries in ~5 min; if that also fails, stop the pod in the console")
else:
    print("HEO_TERMINATE_MODE=none -> pod left RUNNING and BILLING. Stop it yourself when done.")

## RunPod에서 실행 (v4)

```bash
# 팟에 heo_worker.py (v4) 와 이 노트북을 같은 디렉터리(/workspace)에 올린다.
export HEO_WORKDIR=/workspace/heo_v4                      # 신규 (기본값)
export HEO_BASE_WORKDIR=/workspace/heo_v2                 # v3 실행, 읽기 전용 (기본값)
export HEO_BASE_RESULTS=/workspace/heo_v2/results/results_v3.csv
export HEO_X4_SCOPE=anchors                               # 1) 파일럿: 셀 5d만 계산, PASS 3 no-op
export HEO_MODE=orb  HEO_TERMINATE_MODE=none  HEO_IDLE_MIN=180
export MP_API_KEY=...                                     # 셸에서만 (hull 캐시가 base에 있으면 쓰이지 않음)
```

**순서 (스펙 §5)**
1. `HEO_X4_SCOPE=anchors` → 셀 1~5d. 앵커·캘리브레이션·Na 참조는 전부 base 캐시 히트(수 초), 5d의 80구조만 이완(수 분).
   5d 표에서 HOST `transition`, 둘 다 `nonmonotonic` 아님, V_seg>0 확인. **HEO−HOST 순서는 기록만.**
2. `HEO_X4_SCOPE=top500` → 처음부터 Run All. 셀 6·8·9는 캐시 히트로 즉시 지나가고(워커 "0 comps"), PASS 3(10c)만 10,000셀
   (6 GPU × ~1 h). 셀 11 → 12 → 12c(`results/report_v4.md`) → 13.
3. 로컬/스모크: `HEO_MODE=mock HEO_LIMIT=60 HEO_X4_SCOPE=survivors HEO_WATCHDOG=0` (base 없음 → 전부 mock 계산).

**출력을 셀에 전부 남기면서 한번에 돌리기** — 브라우저 Run All은 탭이 연결된 동안만 셀 출력이 쌓인다(끊기면 계산은 계속되지만 그 사이 출력은 셀에서 사라짐).
셀 1이 모든 print를 `$HEO_WORKDIR/results/notebook_stdout.log`에도 기록하지만, **셀 안에** 남기려면 papermill로 돌린다:
```bash
cd /workspace && pip install -q papermill
nohup papermill heo_na_runpod_v4.ipynb heo_na_runpod_v4_RUN.ipynb --cwd /workspace --log-output > pm_v4.log 2>&1 &
tail -f pm_v4.log        # 진행 상황; 워커 로그는 $HEO_WORKDIR/checkpoints/worker_*_p3.log
```
`heo_na_runpod_v4_RUN.ipynb`에 셀별 출력이 실행되는 대로 저장된다(중간에 assert가 나도 거기까지 전부 남음). 끝나면 그 파일을 열어 확인.
브라우저에서 Run All 해도 되지만 그 경우 탭을 닫지 말 것.
배리어(7b/10b/10e)가 워커 종료까지 블로킹하고, 워커가 비정상 종료하면 1회 자동 재기동한다(체크포인트 재개).
워치독은 `HEO_TERMINATE_MODE=stop`이면 셀 13 완료 5분 뒤 또는 180분 유휴 시 팟을 stop 한다.

**base를 절대 건드리지 않는다**: v4는 `heo_v2`를 읽기만 한다. 실행 뒤 `find /workspace/heo_v2 -newer heo_na_runpod_v4.ipynb | head` 가 비어 있어야 한다.

**대조 실행(선택)**: `HEO_MODE=mace HEO_WORKDIR=/workspace/heo_v4_mace HEO_BASE_WORKDIR=/workspace/heo_v2_mace`(v3 MACE 실행이 있을 때)로 같은 노트북 →
`HEO_CONTRAST_RESULTS=/workspace/heo_v4_mace/results/results_v3.csv` 지정 후 셀 11 재실행하면 δ(x)에 `|dE_A − dE_B|`가 x081/x067에도 반영된다.

## 다음 (이 노트북 밖)
* `report_v4.md`의 x_star_class 분포와 dA/dh_perp 분포를 보고 채점에 넣을지 결정 (지금은 전부 보고 열)
* top-30 DFT Tier-1 (`make_dft_inputs.py`) — v3와 동일 목록(채점 불변)
* Kaufman(PRM 2019) 질서 빈자리 패턴은 여전히 미구현